# 🏆 Master Interview Bank — The Architect's Path

*Every interview question from all 11 modules in one place.*

**Scope:** Python Foundations · DSA · SOLID · Design Patterns · System Design · FastAPI · ELK · Database Scaling · Event-Driven Systems · Concurrency · Networking & Security

**Format:** 5–6 deep-dive questions per module with full model answers, a multiple-choice check, and a gotchas checklist.

---
# Python Foundations — Interview Questions

---
## 🏆 Interview Questions — Python Foundations — Interview Questions

*Model answers included. Say your answer aloud before reading.*

# Python Foundations — Interview Questions

> Format: 5 architectural deep-dive questions with answers, a multiple-choice
> knowledge check with an answer key, and a consolidated gotchas list.

---

## Part 1 — Architectural Deep-Dive Questions

### Q1. Why does `def f(x, cache={})` almost always misbehave, and what is actually happening?


**Deep dive.** A default argument is evaluated **once**, at function-definition
time, and that single object is reused on every call that doesn't pass the
argument. So a mutable default (`{}`, `[]`) becomes shared state that accumulates
across calls — the "cache" from one caller silently leaks into the next, and the
bug is invisible until a second call sees the first call's data. The mechanism is
the object/reference model: the default is stored on the function object
(`f.__defaults__`) and rebound to the parameter name each call, never re-created.
The fix is the sentinel idiom: default to `None` and build the fresh object
inside the body (`if cache is None: cache = {}`). The senior point is that this
isn't a quirk to memorize — it's the same mutability-plus-shared-reference fact
that causes aliasing bugs, which is why immutability at boundaries is a design
default, not a preference.

---

### Q2. Is Python pass-by-value or pass-by-reference, and what are the consequences?


**Deep dive.** Neither label fits; Python is **call-by-object-reference** (a.k.a.
call-by-sharing). The function receives a *reference* to the same object the
caller holds, but the parameter name is a new binding. Consequences follow from
mutability: if you **mutate** the argument in place (`lst.append(x)`,
`d[k] = v`), the caller sees it — the object is shared. If you **rebind** the
parameter (`lst = [...]`), the caller sees nothing — you only moved the local
name. So passing a mutable object is an implicit contract about who may mutate
it; passing an immutable one (int, str, tuple, frozen dataclass) is inherently
safe. The design lesson is to be deliberate: return new values instead of
mutating inputs when you don't own them, and prefer immutable types across module
boundaries so callers can't be surprised by aliasing.

---

### Q3. What is the `__eq__` / `__hash__` contract, and what breaks if you violate it?


**Deep dive.** The contract has two clauses. First, **objects that compare equal
must hash equal** (`a == b` ⇒ `hash(a) == hash(b)`); the reverse need not hold
(hash collisions are fine). Second, an object used as a dict key or set member
must be **hashable and effectively immutable for the fields that define
equality** — because the container places it in a bucket derived from its hash.
Violations fail *silently*, which is the danger. Override `__eq__` without
`__hash__` and Python sets `__hash__` to `None`, making instances unhashable — a
loud, early failure. Worse is defining both but inconsistently, or mutating a
key after insertion: the object lands in a bucket, its hash changes, and lookups
that "should" find it return a miss while the entry still occupies memory. That's
why value objects (see `Money` in `advanced_dunder.py`) are immutable and hash on
the same fields they compare on.

---

### Q4. Generators vs lists for a large or unbounded stream — what changes, and what is back-pressure?


**Deep dive.** A list computes and stores every element up front: O(n) memory and
all the work happens before you use the first item. A generator computes one
element per `next()` and suspends its frame in between: O(1) memory, work is
**lazy**, and it can represent an **infinite** sequence a list never could. The
architectural payoff for large streams is that memory stays flat regardless of
length, and you get natural **back-pressure** — because nothing is produced until
the consumer pulls, a slow consumer automatically throttles a fast producer
without any explicit buffer or queue management. The trade-offs are real: a
generator is single-pass (consume it twice and the second pass is empty), it
isn't indexable, and its laziness defers *when* exceptions surface, which can
confuse debugging. Choose a list when you need random access or multiple passes;
choose a generator when the data is large, streamed, or infinite.

---

### Q5. How does CPython free memory, and when is `weakref` the right tool rather than a normal reference?


**Deep dive.** CPython's primary mechanism is **reference counting**: every object
tracks how many references point at it, and it's freed *immediately* when the
count hits zero — deterministic, no pause. Refcounting has one blind spot:
**reference cycles** (A → B → A) keep each other's counts above zero forever, so a
**cyclic garbage collector** runs periodically to detect and reclaim unreachable
cycles. The consequence for design is that anything holding a strong reference
extends an object's lifetime — a cache, an observer list, a parent↔child
back-pattern can leak by keeping objects alive after their real owners are gone.
`weakref` is the right tool exactly there: it lets you *observe* or *cache* an
object **without** contributing to its refcount, so a `WeakValueDictionary`
entry disappears automatically when the last strong owner is dropped. Use it for
caches and back-references you don't want to own; use normal references
everywhere ownership is intended.

---

### Q6. When would you choose a `Protocol` over an ABC to define an interface?


**Deep dive.** Both express "this is the shape a collaborator must have," but they
differ in *how membership is decided*. An **ABC** is **nominal**: a type belongs
only if it explicitly subclasses (or is registered with) the ABC, and it can
provide shared implementation and enforce the contract at instantiation. A
**`Protocol`** is **structural**: any object with the right methods satisfies it,
with no inheritance and no import coupling — the implementer doesn't even need to
know the protocol exists. Prefer a `Protocol` when you're defining a dependency
your code *consumes*, especially across a boundary or when adapting third-party
types you can't subclass — it's how you get Dependency Inversion by shape and keep
tests trivial (pass a plain fake). Prefer an ABC when you own the hierarchy and
want to **share code** in the base or force subclasses to implement methods with a
runtime error. The senior answer names the axis — nominal-with-shared-code vs
structural-and-decoupled — instead of claiming one is universally better.

---

## Part 2 — Multiple-Choice Knowledge Check

**1. `def add(x, items=[]): items.append(x); return items` — calling it three times with only `x` gives:**
- A) three separate one-element lists
- B) a growing shared list because the default is created once
- C) a `TypeError`
- D) an empty list each time

**2. Python's argument passing is best described as:**
- A) pass-by-value (arguments are copied)
- B) pass-by-reference (assigning the parameter changes the caller)
- C) call-by-object-reference (shared object; rebinding is local, mutation is visible)
- D) pass-by-name

**3. If you override `__eq__` on a class but not `__hash__`, instances are:**
- A) still hashable with the default hash
- B) unhashable (`__hash__` set to `None`)
- C) automatically frozen
- D) equal to everything

**4. The main advantage of a generator over a list for a 10-million-row stream is:**
- A) it can be indexed faster
- B) it uses ~constant memory and produces lazily (with back-pressure)
- C) it can be iterated many times
- D) it validates the data

**5. A reference *cycle* between two objects is reclaimed by:**
- A) reference counting alone
- B) the cyclic garbage collector
- C) `weakref`
- D) never — it always leaks

**6. `weakref.WeakValueDictionary` is the right choice when you want to:**
- A) keep cached values alive as long as the cache exists
- B) cache values without preventing them from being garbage-collected
- C) make dictionary access faster
- D) store unhashable keys

### Answer Key
1. **B** — the default list is created once at definition time and shared.
2. **C** — call-by-object-reference: mutation is visible, rebinding is local.
3. **B** — defining `__eq__` without `__hash__` sets `__hash__` to `None`.
4. **B** — constant memory, lazy production, natural back-pressure.
5. **B** — refcounting can't see cycles; the cyclic GC reclaims them.
6. **B** — weak values let entries vanish when the last strong owner is dropped.

---

## Part 3 — Gotchas Checklist

- **Mutable default arguments** (`=[]`, `={}`) are evaluated once and shared —
  default to `None` and build inside the function.
- **Aliasing** — `b = a` on a mutable object shares it; mutate through one and the
  other sees it. Copy or use immutables at boundaries.
- **`__eq__` without `__hash__`** makes instances unhashable; defining both
  inconsistently (or mutating a key) breaks dict/set lookups *silently*.
- **Generators are single-pass and not indexable** — consuming one twice yields
  nothing the second time; materialize to a list only when you truly need reuse.
- **Laziness defers errors** — an exception in a generator surfaces when it's
  pulled, not when it's created; account for this when debugging pipelines.
- **`functools.wraps`** — omit it and your decorator erases the wrapped
  function's name, docstring, and signature, breaking introspection and tooling.
- **Context managers that swallow exceptions** — returning truthy from `__exit__`
  suppresses errors; do it by accident and failures disappear.
- **Reference cycles + strong caches leak** — reach for `weakref` for caches and
  back-references you don't intend to own.
- **`__slots__` and descriptors are optimizations, not defaults** — apply them
  where the memory profile or the invariant justifies the lost flexibility.
- **Type hints don't enforce at runtime** — they power tooling and `Protocol`
  structural typing; validate real data at the boundary (Pydantic), not with hints.

---
# DSA in Production APIs — Interview Questions

---
## 🏆 Interview Questions — DSA in Production APIs — Interview Questions

*Model answers included. Say your answer aloud before reading.*

# DSA in Production APIs — Interview Questions

> Format: 5 architectural questions with deep-dive answers, then a multiple-choice
> knowledge check with an answer key, then a consolidated gotchas list.

---

## Part 1 — Architectural Deep-Dive Questions

### Q1. An endpoint that was fast in staging times out in production. It does `if item in collection` in a loop. What happened and how do you fix it?


**Deep dive.** In staging the collection was small, so the O(n) linear scan of a
`list` was invisible. In production the collection grew (say 50k rows), and
because the loop runs the scan once per requested item (m of them), the endpoint
is O(n·m). At 50k × 500 that's 25M comparisons *per request*, executed
synchronously — it saturates CPU and, in an async app, blocks the event loop so
*all* concurrent requests stall.

The fix is choosing the right structure: a `set`/`frozenset` gives O(1) average
membership, dropping the endpoint to O(n + m). The index (set) should be built
**once** (at startup or cached), not rebuilt per request. The deeper lesson is
that algorithmic complexity is a *production latency budget*: always ask "what is
n at scale, and what bounds it?" A structure that's fine at n=100 can be an
outage at n=10⁶.

---

### Q2. How does an unbounded request body become a denial-of-service vector, even without malicious intent?


**Deep dive.** If the endpoint accepts `list[str]` with no cap, the client
controls `m`. Combine that with an O(n) operation per element and the server-side
cost scales as O(n·m) under client control — an **algorithmic complexity attack**
(amplification). Even a well-meaning client running a large batch can trigger it.
The defense is to bound N at the trust boundary (Pydantic `max_length`), so the
worst-case cost is provably capped. Bounds on input size, page size, and nesting
depth are architectural decisions, not afterthoughts. Related: unbounded response
sizes (returning all rows) are the mirror problem — always paginate.

---

### Q3. When is an O(n²) algorithm inside an endpoint acceptable?


**Deep dive.** When N is small *and provably bounded*. If the input is a config
list capped at 50 items, an O(n²) nested loop is ~2,500 operations — negligible,
and often more readable than a cleverer structure. Premature optimization adds
risk (bugs, complexity) for no measurable gain. The senior signal is refusing to
optimize *or* to leave it slow without first establishing the bound on N and the
latency budget. "It depends on what bounds N" is the correct opening, not a
reflexive "always use a hash map."

---

### Q4. You need an autocomplete endpoint. Compare scanning a list, a SQL `LIKE`, and a trie.


**Deep dive.** *List scan / `startswith` over all rows* is O(n) per keystroke —
fine for thousands, hopeless for millions and it repeats every keystroke.
*SQL `LIKE 'prefix%'`* can use a B-tree index (prefix matches are
range scans) and is a reasonable default that offloads work to the database.
*Trie* gives O(k) prefix lookup independent of the number of keys (k = prefix
length), ideal for very high-QPS autocomplete, at the cost of memory and having
to keep the trie in sync with the source of truth. The choice is a trade-off of
QPS, dataset size, memory, and operational simplicity — for most apps a
DB-indexed `LIKE` or a search engine (Elasticsearch) beats hand-rolling a trie.

---

### Q5. Why is caching a computed result in a `dict` both a performance win and a correctness risk?


**Deep dive.** A `dict` (or Redis) cache converts an O(n) or O(log n) lookup into
O(1), which is a large win for read-heavy endpoints. But a cache is a **second
source of truth**, so it introduces invalidation — the hard problem. Risks: stale
entries when the source changes, unbounded memory growth without an eviction
policy (use an LRU/TTL), and the **thundering herd / cache stampede** when a hot
key expires and many requests recompute it at once. Senior caching always
specifies capacity, eviction, TTL, and a stampede mitigation (single-flight lock
or probabilistic early expiry) — not just "add a dict."

---

## Part 2 — Multiple-Choice Knowledge Check

**1. What is the time complexity of `x in my_list` for a Python `list`?**
- A) O(1)
- B) O(log n)
- C) O(n)
- D) O(n log n)

**2. Doing a list-membership test for `m` items against a list of `n` items is:**
- A) O(n + m)
- B) O(n · m)
- C) O(m log n)
- D) O(1)

**3. The best fix to make repeated membership tests O(1) average is to use a:**
- A) sorted list + linear scan
- B) `set` / `frozenset`
- C) tuple
- D) generator

**4. Accepting an unbounded `list` in a request body primarily risks:**
- A) a SQL injection
- B) an algorithmic-complexity denial of service
- C) a CSRF attack
- D) a memory leak in the client

**5. An O(n²) algorithm inside an endpoint is acceptable when:**
- A) never — always optimize
- B) the input size N is small and provably bounded
- C) only in staging
- D) the endpoint is a GET

### Answer Key
1. **C** — list membership is a linear scan.
2. **B** — m scans × O(n) each = O(n·m).
3. **B** — a set gives O(1) average membership.
4. **B** — unbounded N + per-element cost = algorithmic DoS.
5. **B** — a small, bounded N makes O(n²) negligible and often clearer.

---

## Part 3 — Gotchas Checklist

- **`in` on a list is O(n).** Use a `set`/`dict` for membership and keyed lookup.
- **Build the index once.** Rebuilding a set/dict per request throws away the win.
- **Bound every input.** `max_length` on lists, page-size caps, nesting limits —
  answer "what bounds N?" at the boundary.
- **Bound every output.** Returning all rows is the mirror DoS; paginate.
- **Hash-map worst case is O(n).** "O(1)" is *amortized/average*; pathological
  collisions (or untrusted keys) degrade it — usually fine, but know it exists.
- **Recursion depth ∝ input** is a crash vector (Python ~1000-frame limit); use
  iterative traversal on untrusted/deep data.
- **CPU-bound loops block the event loop** in `async` routes — offload or bound.
- **A cache is a second source of truth**: set capacity, eviction, TTL, and a
  stampede guard, or you trade a speed bug for a correctness bug.

---
# SOLID in FastAPI — Interview Questions

---
## 🏆 Interview Questions — SOLID in FastAPI — Interview Questions

*Model answers included. Say your answer aloud before reading.*

# SOLID in FastAPI — Interview Questions

> Format: 5 architectural questions with deep-dive answers, a multiple-choice
> knowledge check with an answer key, and a consolidated gotchas list.

---

## Part 1 — Architectural Deep-Dive Questions

### Q1. A route function does validation, business logic, DB access, a payment call, and email. Which SOLID principles does it violate and why does it matter?


**Deep dive.** It violates at least three. **SRP** — the function has five reasons
to change (validation rules, pricing, schema, payment API, notification), so a
change to any one risks breaking the others; changes *collide* in one place.
**OCP** — provider selection via `if/elif` means adding PayPal edits tested code.
**DIP** — it hard-codes concrete details (the DB, the gateway), so the business
logic is welded to the web framework and can only be tested by spinning up HTTP
and patching globals. Why it matters: the code becomes slow to change, risky to
modify, and nearly impossible to unit-test — the three properties that most
determine a codebase's cost over time.

---

### Q2. Explain how Dependency Inversion turns an untestable route into a fast unit test.


**Deep dive.** DIP says high-level policy depends on abstractions, not concrete
details. By extracting a `RegistrationService` that receives a `UserRepository`,
`Notifier`, and `PaymentGateway` (all Protocols) via its constructor, the service
never imports FastAPI and never constructs its own dependencies. In a test you
instantiate it with in-memory fakes and call `register()` directly — no server,
no database, milliseconds per test. In production, FastAPI's `Depends` wires the
real implementations at the composition root. The abstraction is the *seam* that
makes both substitution (swap Stripe for PayPal) and isolation (fake in tests)
possible.

---

### Q3. Where exactly should the "composition root" live in a FastAPI app, and why does it matter?


**Deep dive.** The composition root is the single place where abstractions are
bound to concrete implementations — in FastAPI, the `Depends` provider functions
(`get_service`, `get_gateway`) and the `lifespan` handler. It must sit at the
*edge* of the system so the inner layers (service, domain) stay ignorant of which
concrete DB or gateway is used. This matters because it localizes the "which
implementation?" decision: swapping in-memory for Postgres, or Stripe for a mock,
is a one-line change there and nowhere else. Scattering `new StripeGateway()`
through the code destroys that property and re-couples everything.

---

### Q4. Your teammate replaces the `if/elif` provider ladder with a dict of gateways. Is that enough to satisfy OCP?


**Deep dive.** It's a big step, but "enough" depends on the *registration*
mechanism. A dict lookup replaces the conditional, and adding a provider means
adding a class + a dict entry rather than editing branching logic — that's the
spirit of OCP. It's fully realized when new providers can be *registered* without
editing the dict's definition either (e.g., a plugin registry or entry-points), so
the core module is truly closed to modification. For most apps the dict is the
pragmatic sweet spot; over-engineering a plugin system for two providers is
YAGNI. The senior answer names the trade-off rather than claiming a single right
answer.

---

### Q5. When does applying SOLID become over-engineering in a FastAPI service?


**Deep dive.** When you introduce abstractions with a single implementation and no
test seam — e.g., a `Protocol` and a factory for a value that is stable and only
ever built one way. Every abstraction is indirection a reader must hold in their
head, and FastAPI already gives you DI cheaply, which tempts over-layering. The
heuristic: introduce an interface when there's a genuine second implementation, a
volatile external boundary (payment, email, storage), or a testing seam you
actually use. A CRUD endpoint over one table doesn't need three layers and four
Protocols. SOLID controls coupling; if there's no coupling worth controlling, the
abstraction is pure cost.

---

## Part 2 — Multiple-Choice Knowledge Check

**1. A route function that validates, saves to DB, charges a card, and sends email violates primarily:**
- A) Liskov Substitution
- B) Single Responsibility
- C) Interface Segregation
- D) none — it's fine

**2. Selecting a payment provider with `if provider == 'stripe' elif ...` violates:**
- A) Open/Closed Principle
- B) Liskov Substitution
- C) DRY only
- D) nothing

**3. In FastAPI, the mechanism that provides Dependency Inversion is:**
- A) middleware
- B) `Depends()`
- C) `BackgroundTasks`
- D) `response_model`

**4. The main testability benefit of extracting a framework-free service layer is:**
- A) it runs faster in production
- B) business logic can be unit-tested without HTTP or a database
- C) it reduces the number of files
- D) it removes the need for Pydantic

**5. Introducing a Protocol with exactly one implementation and no test seam is usually:**
- A) required by SOLID
- B) speculative generality / over-engineering
- C) a Liskov violation
- D) necessary for OCP

### Answer Key
1. **B** — five reasons to change = SRP violation.
2. **A** — a type-switch that grows requires editing tested code (OCP).
3. **B** — `Depends()` is dependency injection built into FastAPI.
4. **B** — a framework-free service is testable in isolation.
5. **B** — abstraction without a second impl or a test seam is YAGNI.

---

## Part 3 — Gotchas Checklist

- **"God routes" fuse five concerns.** Keep routers thin: translate HTTP ⇄ domain
  and map errors to status codes; push logic to a service.
- **`if/elif` on a type code** is an OCP smell — replace with polymorphism or a
  registry/dict of strategies.
- **Business logic importing FastAPI** is the tell that it can't be unit-tested;
  the service layer must be framework-free.
- **Hidden global state** (a module-level dict, a singleton) makes tests leak into
  each other — inject dependencies instead.
- **Validate at the boundary, not everywhere.** Pydantic at the edge means inner
  layers assume clean data (don't re-validate in every function).
- **Return DTOs, not DB/domain models**, or you leak internal fields (a password
  hash) and couple your wire format to your schema.
- **Composition root at the edge only.** Constructing concretes deep in the code
  re-couples layers and defeats DIP.
- **Over-layering is also a smell.** One implementation + no test seam = delete
  the abstraction. SOLID is coupling control, not a checklist to maximize.

---
# Design Patterns in FastAPI — Interview Questions

---
## 🏆 Interview Questions — Design Patterns in FastAPI — Interview Questions

*Model answers included. Say your answer aloud before reading.*

# Design Patterns in FastAPI — Interview Questions

> Format: 5 architectural questions with deep-dive answers, a multiple-choice
> knowledge check with an answer key, and a consolidated gotchas list.

---

## Part 1 — Architectural Deep-Dive Questions

### Q1. A notification endpoint has a growing `if channel == ...` ladder. Which pattern fixes it and what exactly does it buy?


**Deep dive.** The Strategy pattern: define a `NotificationChannel` interface and
one implementation per channel, then select at runtime via a Factory (a dict
mapping name → strategy). What it buys is Open/Closed compliance — adding a
channel is a new class plus a registry entry, editing no tested branching logic —
and testability, since each channel is an isolated unit and the route no longer
contains behavior. It also removes the risk that a change to the SMS branch
accidentally breaks the email branch, because they no longer share a function.
The trade-off is a little indirection, justified once the set of channels is
expected to grow.

---

### Q2. Explain the circuit breaker's three states and how it prevents a cascading failure.


**Deep dive.** **CLOSED** — calls pass through and failures are counted. After N
consecutive failures the breaker trips to **OPEN** — calls fail *immediately* for
a cooldown window, without touching the dead dependency. After the cooldown it
moves to **HALF-OPEN** and allows one probe: success closes it, failure reopens
it. It prevents cascading failure by converting slow timeouts into fast
rejections: when a downstream provider is down, threads/connections aren't held
waiting on doomed calls, so the calling service doesn't exhaust its own resources
and take *itself* down. The breaker trades a brief period of shedding load for
overall system survival, and gives the dependency room to recover.

---

### Q3. How do retries, timeouts, and circuit breakers combine, and what's the danger of using retries alone?


**Deep dive.** They form a layered resilience strategy. **Timeouts** bound how
long any single call can hang (mandatory on every network call). **Retries with
exponential backoff + jitter** recover from *transient* blips. **Circuit
breakers** handle *sustained* outages by stopping retries entirely. Retries alone
are dangerous: during a real outage, aggressive retries multiply load on an
already-failing dependency (a "retry storm"), and synchronized retries create a
thundering herd. Backoff + jitter de-synchronizes them, and the breaker caps the
damage by short-circuiting once failures are clearly not transient. The senior
design uses all three together.

---

### Q4. Someone submits a PR adding a `FactoryFactory` and five interfaces to send an email. How do you respond?


**Deep dive.** Push back with the cost/benefit lens. A pattern earns its place
only when it removes more complexity (coupling, duplication, rigidity) than the
indirection it adds. If there's one implementation, no test seam, and no
foreseeable second variant, that's speculative generality (YAGNI) — it makes the
code harder to read now for a benefit that may never arrive. I'd suggest
collapsing it to a plain function or a single Strategy interface and
re-introducing abstraction when a real second channel appears. Recognizing
*over*-patterning is as much a senior skill as knowing the patterns.

---

### Q5. Where does the Adapter pattern belong in a service that integrates a third-party SDK?


**Deep dive.** At the boundary. Wrap the vendor SDK in an Adapter that conforms to
*your* interface (e.g., a `NotificationChannel`/`Notifier` you define), so your
application code depends on your abstraction rather than the vendor's signatures.
The payoff: when you switch vendors or the SDK changes, you write/modify one
adapter and nothing else changes; and in tests you substitute a fake implementing
your interface without mocking the vendor's concrete classes. This is Dependency
Inversion at an integration point, and it keeps third-party churn from rippling
through your codebase. Pair it with a circuit breaker for the network call.

---

## Part 2 — Multiple-Choice Knowledge Check

**1. Replacing a growing `if channel == ...` ladder with interchangeable classes is the:**
- A) Singleton pattern
- B) Strategy pattern
- C) Decorator pattern
- D) Visitor pattern

**2. A circuit breaker in the OPEN state will:**
- A) retry the call indefinitely
- B) fail fast without calling the dependency
- C) cache the last successful response
- D) increase the timeout

**3. Using retries WITHOUT backoff during an outage tends to cause:**
- A) a retry storm that worsens the outage
- B) a memory leak
- C) a SQL injection
- D) nothing — it's best practice

**4. Wrapping a third-party SDK behind your own interface is the:**
- A) Facade pattern
- B) Adapter pattern
- C) Observer pattern
- D) Factory pattern

**5. A `FactoryFactory` with five interfaces to construct one stable object is:**
- A) required by the Gang of Four
- B) over-engineering (YAGNI)
- C) the Strategy pattern
- D) a circuit breaker

### Answer Key
1. **B** — interchangeable algorithms behind one interface = Strategy.
2. **B** — OPEN means fail fast, no call to the dependency.
3. **A** — retries without backoff amplify load (retry storm).
4. **B** — conforming a foreign interface to yours = Adapter.
5. **B** — abstraction with no second impl/test seam = over-engineering.

---

## Part 3 — Gotchas Checklist

- **Every remote call needs a timeout.** No timeout = a hung thread waiting on a
  dead dependency; enough of them and the service dies.
- **Retries need backoff + jitter.** Naive retries synchronize into a thundering
  herd and turn a blip into an outage (retry storm).
- **A breaker without a HALF-OPEN probe never recovers** — or recovers by
  slamming the dependency with full traffic. Probe with one call.
- **Tune breaker thresholds to the dependency.** Too sensitive = false trips; too
  lax = no protection. Base it on real error-rate/latency SLOs.
- **Strategy/Factory registries can hide typos** — an unknown key must fail
  loudly (400/validation), not silently no-op.
- **Don't confuse Adapter and Facade.** Adapter changes an incompatible shape;
  Facade simplifies a complex subsystem. Using the wrong word signals confusion.
- **Over-patterning is a real anti-pattern.** Indirection you can't justify with a
  concrete need is cost, not craftsmanship.
- **Singletons for shared state** hide coupling and break tests — prefer DI.

---
# System Design in FastAPI — Interview Questions

---
## 🏆 Interview Questions — System Design in FastAPI — Interview Questions

*Model answers included. Say your answer aloud before reading.*

# System Design in FastAPI — Interview Questions

> Format: 5 architectural questions with deep-dive answers, a multiple-choice
> knowledge check with an answer key, and a consolidated gotchas list.

---

## Part 1 — Architectural Deep-Dive Questions

### Q1. A "charge card" endpoint sometimes double-charges customers. What's the root cause and the fix?


**Deep dive.** The network guarantees *at-least-once* delivery: clients, load
balancers, and proxies all retry on timeout, and a request can succeed on the
server while the *response* is lost — the client then retries a charge that
already happened. With nothing to deduplicate on, the retry charges again. The
fix is an **idempotency key**: the client sends a unique key per logical
operation; the server performs the charge the first time and stores the response
keyed by it, then *replays* that stored response on any retry with the same key,
without re-charging. This converts at-least-once delivery into exactly-once
*effect*. Critically, persist the result *before* acknowledging, so a
crash-then-retry is still safe.

---

### Q2. Why is doing slow work synchronously on the request path an architectural problem, and what are the options for moving it off?


**Deep dive.** Slow inline work (PDF render, email, third-party call) inflates p99
latency, holds the worker/connection for its full duration (reducing throughput),
and couples the client's success to the slow task's success. Options, in
increasing robustness: **FastAPI `BackgroundTasks`** (simple, in-process — runs
after the response, but is lost if the process dies and doesn't survive restarts);
a **task queue** (Celery/RQ/Arq — durable, retryable, scalable across workers); or
**publishing an event** to a broker for a separate consumer (fully decoupled,
event-driven). The right choice depends on durability needs: `BackgroundTasks` for
best-effort fire-and-forget, a queue/broker when the work *must* eventually happen.

---

### Q3. Apply CAP to this payment service. During a network partition, what do you choose?


**Deep dive.** Payments demand correctness, so you choose **CP** (consistency over
availability): during a partition, it's better to reject or hold a transaction
than to risk a double-spend or a divergent ledger that must later be reconciled by
hand. That means the write path depends on a strongly-consistent store and will
return errors rather than accept ambiguous writes when it can't guarantee
correctness. The complement is PACELC: even without a partition, you're trading
latency for consistency — synchronous strong consistency costs p99 latency on
every charge, which you accept for money-handling. Contrast with a product-view
counter, where you'd pick AP and tolerate staleness.

---

### Q4. Where should the idempotency store live, what's its lifecycle, and what are the failure modes?


**Deep dive.** It should be a fast, shared, durable store — typically Redis or a
DB table — *shared across all instances*, because a per-process dict doesn't
deduplicate across a fleet (a retry hitting a different instance would re-charge).
Entries carry a **TTL** matched to the retry window (hours, not forever) to bound
memory. Failure modes: (a) storing the response *after* acking, so a crash between
charge and store loses the dedupe record — persist before returning; (b) racing
concurrent retries with the same key — use an atomic set-if-absent (`SET NX`) or a
DB unique constraint so only one wins; (c) storing a *failed* attempt as success —
be deliberate about whether errors are cached.

---

### Q5. Do a back-of-the-envelope estimate for this service (5M charges/day) and identify the bottleneck.


**Deep dive.** 5M / 86,400 ≈ 58 charges/sec average; design for peak at 3–5×, so
~200–300 writes/sec. That's a modest write rate a single well-tuned primary
handles, so the bottleneck is unlikely to be raw DB write throughput — it's more
likely the **synchronous downstream work** (payment provider latency, receipt
generation) inflating latency and tying up workers, plus correctness under retry.
The math tells you the design priorities here are *idempotency and offloading slow
work*, not sharding. The point of the estimate is to reveal that the scaling
problem is latency/correctness, not volume — so you don't over-engineer storage.

---

## Part 2 — Multiple-Choice Knowledge Check

**1. Networks provide which delivery guarantee by default, forcing idempotency?**
- A) exactly-once
- B) at-most-once
- C) at-least-once
- D) ordered-once

**2. An idempotency key makes a retried charge safe by:**
- A) encrypting the request
- B) replaying the stored response without re-executing the side effect
- C) rejecting all retries
- D) charging a smaller amount

**3. You must persist the idempotency result:**
- A) after returning the response
- B) before acknowledging the response
- C) only on failure
- D) never — keep it in memory

**4. A per-process in-memory idempotency dict fails in production because:**
- A) it's too slow
- B) it doesn't deduplicate across multiple server instances
- C) it violates SOLID
- D) dicts can't store responses

**5. For a payment service during a network partition, you choose:**
- A) AP — availability over consistency
- B) CP — consistency over availability
- C) neither — CAP doesn't apply
- D) both simultaneously

### Answer Key
1. **C** — at-least-once delivery is why retries duplicate.
2. **B** — replay the stored result; run the side effect once.
3. **B** — persist before acking so crash-then-retry is safe.
4. **B** — a local dict can't dedupe across a fleet; use a shared store.
5. **B** — payments favor correctness (CP).

---

## Part 3 — Gotchas Checklist

- **Assume at-least-once delivery.** Any endpoint with a side effect (charge,
  ship, post) needs an idempotency key or naturally-idempotent operation.
- **Persist the idempotency record BEFORE responding**, or a crash between the
  effect and the store re-runs the effect on retry.
- **Use a shared, TTL'd idempotency store** (Redis/DB), not a per-process dict —
  otherwise retries to other instances duplicate.
- **Guard concurrent retries** with an atomic set-if-absent / unique constraint,
  or two in-flight retries both execute.
- **`BackgroundTasks` are best-effort** and in-process: they run after the
  response but are lost on crash/restart. Use a durable queue when the work must
  happen.
- **Don't do slow/CPU-bound work on the request path** — it inflates p99 and
  starves workers.
- **Do the capacity math first** — it tells you whether the problem is volume
  (shard) or latency/correctness (idempotency + offload), preventing
  over-engineering.
- **Decide error caching explicitly** — caching a transient failure as the
  permanent result under an idempotency key is a nasty, subtle bug.

---
## Part 4 — Advanced Architecture (Q6–Q14)
*(Scalability · HA · Quorum · CAP · Connection Pooling · Leader Election · Consensus · Backpressure · Idempotency · DLQ · Saga · API Gateway · Service Discovery)*

### Q6. ShopFlow hits 50k RPS. The CTO says "just get a bigger DB server." What's wrong, and what's your counter-proposal?

**Deep dive.** Vertical scaling has three hard limits: exponential cost past mid-tier, a hardware ceiling (~96 cores max), and a single point of failure. At 50k RPS the bottleneck is almost always **read-heavy** traffic, not write throughput. Counter-proposal in order: (1) vertical (free, no code change); (2) **read replicas** — route SELECTs there; (3) **Redis cache** in front of hot read paths (often cuts DB load 80%); (4) **sharding** by user_id/region only if writes are the bottleneck. Critical prerequisite: the app tier must be **stateless** (sessions in Redis, not server RAM) before horizontal scaling works. The traffic shape (read/write ratio, hot keys) tells you which lever to pull.

---

### Q7. Your service has three DB replicas. A partition isolates one. What happens? Why does "always use odd nodes" matter?

**Deep dive.** Quorum = ⌊N/2⌋+1. With N=3, quorum=2. The majority side (2 nodes) reaches quorum → continues writes. The isolated node (1 node) cannot → refuses writes, preventing split-brain. Odd-node rule: N=4 has quorum=3, same fault tolerance as N=3 (both survive 1 failure), but costs an extra server. N=4 buys nothing. Always use 3, 5, or 7. Synchronous replication to ≥1 replica for money writes; async for analytics. PACELC: even without partitions, synchronous consistency adds latency on every write — name that tradeoff in every design review.

---

### Q8. Explain CAP. Where do your datastores live on the triangle, and how does it affect design decisions?

**Deep dive.** CAP: during a network **P**artition choose **C**onsistency (all nodes see same data) OR **A**vailability (every request gets a response, possibly stale). You can't have both. Practical mapping: **PostgreSQL (CP)** — primary stops writes if it can't confirm replicas; **Cassandra (AP)** — every node always accepts writes (eventual consistency); **Redis Sentinel (CP)** — majority quorum for primary election. Design rule: use CP (Postgres) for money/orders/inventory; use AP (Cassandra/DynamoDB) for views, activity, metrics. PACELC refines this: even without partitions Cassandra trades Latency for Consistency. Name the tradeoff in every design review — never pick a store without saying which side you're on.

---

### Q9. A service opens 1,000 simultaneous DB connections at peak. How do you fix it and size the pool?

**Deep dive.** Each PostgreSQL connection = ~5–10MB RAM + a backend process. 1,000 connections = 5–10GB just for overhead → OOM → crash. Fix: **connection pooling** — a fixed-size pool of N reusable connections shared across all threads. Sizing formula: `pool_size ≈ num_CPU_cores × 2 + num_effective_spindles` (PostgreSQL's recommendation). 4-core DB → pool_size ≈ 10 per app instance; 20 app servers → 200 total — must be under `max_connections / 2` (leave headroom for admin). Production standard: **PgBouncer** as a proxy pool — it multiplexes thousands of app connections onto tens of DB connections transparently. For async (asyncpg), pools can be smaller since connections aren't blocked waiting for I/O.

---

### Q10. Your Kafka consumer crashes after processing a message but before committing the offset. What happens, and why does your handler need to be idempotent?

**Deep dive.** Kafka re-delivers the uncommitted message on restart (at-least-once delivery). If the handler has side effects (charge card, send email), the side effect runs twice. Three idempotency strategies: (a) **natural idempotency** — `UPDATE SET status='processed' WHERE status='pending'` is inherently safe to repeat; (b) **idempotency key** — store `(event_id, result)` in Redis with `SET NX`; return cached result on retry without re-executing; (c) **deduplication table** — persist processed event_id in the same DB transaction as the business write. DLQ: if a message can never be processed (poison pill), after N retries move it to DLQ to unblock the partition — never silently drop.

---

### Q11. Design the leader election for a distributed cron scheduler. What prevents two nodes from both believing they're the leader?

**Deep dive.** The zombie leader problem: an old leader pauses (GC, network hiccup), a new leader is elected, the old leader resumes and believes it's still in charge — two writers. Prevention: **fencing tokens** — every election issues a monotonically increasing integer; storage rejects writes with a stale (lower) token. Raft mechanics: (1) follower times out → becomes candidate → increments term → sends RequestVote to peers; (2) majority votes YES on same term → wins; (3) any node seeing a higher term immediately reverts to follower. Terms are logical clocks. Production: use **etcd `LeaseGrant`** (TTL-based leader lock) + `KeepAlive` heartbeat — never roll your own consensus.

---

### Q12. A payment worker's queue depth grows faster than it drains. What is backpressure and what are your options?

**Deep dive.** Backpressure = resistance applied upstream when downstream can't keep up. Without it: unbounded queue growth → OOM → crash. Options ranked by data-safety: (1) **Block** — producer blocks when queue is full; correct for payments, propagates delay; (2) **Drop** — reject new items at capacity; acceptable for metrics/telemetry, never for payments; (3) **Sample** — accept 1-in-N; for high-volume logging; (4) **Scale out** — autoscale consumers on queue depth (HPA); correct but has lag; (5) **Circuit breaker** — surface 503 to callers. For payments: block + autoscale + alert. Key gotcha: **unbounded queues hide the problem** — app runs fine until OOM. In asyncio use `asyncio.Queue(maxsize=N)` with `await q.put()` — blocking `queue.Queue.put()` blocks the event loop.

---

### Q13. Checkout touches inventory, payment, and fulfillment. Payment succeeds, order creation fails. How do you roll back?

**Deep dive.** You cannot use distributed 2PC across services — it holds locks on all services simultaneously; a coordinator crash leaves everyone blocked forever. Use the **Saga pattern**: a sequence of local transactions where on failure, completed steps are **compensated in reverse**. Example: Reserve Inventory → Charge Payment → Create Order [FAILS] → Compensate: Refund Payment → Release Inventory. Compensation is NOT a DB ROLLBACK — it's new business logic (a Stripe refund API call, a new inventory UPDATE). Two styles: **orchestration** (central saga orchestrator, clear flow, auditable, but orchestrator is a SPOF); **choreography** (services react to events, loose coupling, but failure flows are hard to trace). Gotcha: sagas have no isolation — concurrent sagas can read data that another saga is about to compensate.

---

### Q14. What are the responsibilities of an API gateway, and why does it exist as a separate layer?

**Deep dive.** API gateway = single entry point for all external traffic. Responsibilities: (1) **routing** — map paths to services; (2) **auth** — validate JWT/API key once at the edge, so every microservice doesn't re-implement auth; (3) **rate limiting** — enforce per-user quotas before requests hit services; (4) **SSL termination** — decrypt HTTPS once; (5) **request transformation** — inject headers, strip sensitive fields; (6) **observability** — one place to log latency, inject trace IDs. Without it: 10 microservices = 10 auth implementations, all potentially inconsistent. The gateway also enables gradual deploys (5% to v2) and A/B testing. Service discovery integrates here — the gateway queries the registry for healthy instances of each service.

---

## Part 5 — Knowledge Check (Advanced Topics)

**1.** N=4 nodes, quorum=3. How many failures can it survive?  **A)** 2  **B)** 3  **C)** 1  **D)** 0

**2.** The zombie leader problem is best prevented by:  **A)** Longer heartbeat intervals  **B)** Fencing tokens rejected by storage if stale  **C)** 2PC instead of Raft  **D)** Restarting the old leader

**3.** A Saga's compensating transaction is:  **A)** A database ROLLBACK  **B)** New business logic that explicitly undoes the step (e.g., a refund API call)  **C)** A 2PC prepare phase  **D)** An idempotency key lookup

**4.** Backpressure "drop" strategy is appropriate for:  **A)** Payment processing  **B)** Order creation  **C)** High-volume telemetry where some data loss is acceptable  **D)** DB writes

**5.** An idempotency key in Redis must use which atomic operation?  **A)** GET then SET  **B)** SET NX (set-if-not-exists)  **C)** INCR  **D)** LPUSH

### Answer Key
1. **C** — quorum=3, same as N=3; N=4 wastes a server with no added fault tolerance.
2. **B** — fencing tokens; storage rejects stale-token writes from the zombie leader.
3. **B** — compensation is new business logic, not a DB rollback.
4. **C** — drop only when losing individual data points doesn't affect correctness.
5. **B** — `SET NX` is atomic; GET+SET has a race condition where two retries both execute.


---
# FastAPI Advanced — Interview Questions

---
## 🏆 Interview Questions — FastAPI Advanced — Interview Questions

*Model answers included. Say your answer aloud before reading.*

# FastAPI Advanced — Interview Questions

> Format: 5 architectural questions with deep-dive answers, a multiple-choice
> knowledge check with an answer key, and a consolidated gotchas list.

---

## Part 1 — Architectural Deep-Dive Questions

### Q1. A developer puts `time.sleep(2)` (or a blocking `requests.get`) inside an `async def` route. Describe exactly what happens under load.


**Deep dive.** FastAPI serves `async def` routes on a single event loop per
worker. A blocking call doesn't yield control back to the loop, so for its entire
duration the loop can process *no other requests* — including a trivial
`/health`. Under concurrency this serializes everything: 100 clients hitting a
2-second blocking route experience up to ~200 seconds of tail latency, and the
service appears "hung" even though CPU may be idle. It passes local testing
because a single sequential request never reveals the contention. The tell is that
throughput doesn't scale with concurrency and unrelated endpoints slow down when
one endpoint is busy.

---

### Q2. Give three correct ways to fix a blocking call in a FastAPI app and when to use each.


**Deep dive.** (1) **Use an async library and `await` it** — the ideal fix for
I/O: `httpx.AsyncClient`, `asyncpg`, async SQLAlchemy. The call yields to the loop
while waiting. (2) **Offload to a threadpool** with `run_in_threadpool` (or
`asyncio.to_thread`) when you're stuck with a synchronous library — the blocking
work runs on a worker thread, freeing the loop. (3) **Define the route as plain
`def`** — FastAPI automatically runs sync routes in its threadpool, so legacy
blocking handlers don't block the loop; simplest fix when the whole handler is
synchronous. For **CPU-bound** work, threads don't help much (the GIL) — use a
process pool or, better, push it to a background worker/queue.

---

### Q3. Why is the layered architecture (router → service → repository) worth the extra files in FastAPI?


**Deep dive.** It maps directly to testability and change isolation. The
**service** layer holds business logic and imports no framework, so it's
unit-testable in milliseconds without HTTP or a database, and reusable from a CLI
or worker. The **repository** is an abstraction over storage, so tests inject an
in-memory implementation and production injects Postgres — swapping either is a
one-line change at the composition root (DIP). The **router** stays thin:
translate HTTP to domain calls and domain errors to status codes. Fat routes fuse
HTTP, business, and persistence concerns, so every change is risky and every test
needs a live server. The extra files buy speed of change and speed of testing.

---

### Q4. Explain the role of `lifespan` and dependency injection in resource management.


**Deep dive.** Expensive resources — DB connection pools, HTTP clients, broker
producers — must be created **once** and shared, not per request, or you exhaust
the database's connection limit and add latency. The `lifespan` async context
manager acquires them at startup and releases them at shutdown (enabling graceful
drain). `Depends` then injects those shared resources (or a per-request DB session
derived from the pool) into routes, and lets you override them in tests via
`app.dependency_overrides`. Together they give correct lifecycle management and a
testing seam. The old per-request `create_engine()`/new-connection pattern is a
classic scaling bug.

---

### Q5. Why separate Pydantic input, output, and DB models, and how does this relate to security?


**Deep dive.** Separate models decouple the public API contract from internal
storage and control exposure. The **input** model is the validation/trust boundary
— data is validated and coerced at the edge so inner layers assume clean input.
The **output** (response) model determines exactly what's serialized, preventing
accidental leakage of sensitive fields (password hashes, internal flags, admin
booleans) that returning a DB object directly would expose. And keeping them
separate lets storage evolve (migrations) without breaking clients, and vice
versa. It's decoupling *and* a security control (explicit allow-list of exposed
fields via `response_model`), not mere duplication.

---

## Part 2 — Multiple-Choice Knowledge Check

**1. A blocking `time.sleep()` inside an `async def` route:**
- A) only slows that one request
- B) blocks the entire event loop, stalling all concurrent requests
- C) is automatically offloaded by FastAPI
- D) raises an exception

**2. The idiomatic way to run a synchronous library call from an async route is:**
- A) call it directly
- B) `await run_in_threadpool(fn)` / `asyncio.to_thread`
- C) wrap it in a try/except
- D) add more workers only

**3. A plain `def` route in FastAPI is:**
- A) rejected at startup
- B) run in a threadpool so it doesn't block the loop
- C) run on the event loop like async
- D) slower than async always

**4. Expensive resources like DB pools should be created:**
- A) per request
- B) once, in `lifespan`, and shared via DI
- C) in every dependency
- D) at import time in a global with no cleanup

**5. A separate response_model prevents:**
- A) SQL injection
- B) leaking sensitive internal fields in the response
- C) event-loop blocking
- D) replication lag

### Answer Key
1. **B** — one event loop; blocking it stalls everything.
2. **B** — offload blocking work to a threadpool.
3. **B** — FastAPI runs sync routes in its threadpool.
4. **B** — create once in `lifespan`, inject via `Depends`.
5. **B** — response_model is an explicit output allow-list.

---

## Part 3 — Gotchas Checklist

- **Never block the event loop.** No `time.sleep`, `requests`, sync DB drivers, or
  heavy CPU directly inside `async def`. Use async libs, `run_in_threadpool`, or a
  `def` route.
- **CPU-bound work isn't fixed by threads** (GIL) — use a process pool or a
  background worker/queue.
- **`async def` + a sync DB driver is a trap** — either use an async driver or
  make the route `def`. Mixing them silently blocks the loop.
- **Create pools/clients once in `lifespan`**, not per request; close them on
  shutdown for graceful drain.
- **Don't return ORM/DB models directly** — use a `response_model` to avoid
  leaking sensitive fields and to decouple wire format from schema.
- **Validate at the boundary** with Pydantic; don't re-validate everywhere.
- **`BackgroundTasks` run in-process** and are lost on crash — use a durable queue
  when the work must complete.
- **Blocking bugs hide in local testing** — always load-test with concurrency, and
  watch whether unrelated endpoints slow down together.

---
# ELK / Observability in FastAPI — Interview Questions

---
## 🏆 Interview Questions — ELK / Observability in FastAPI — Interview Questions

*Model answers included. Say your answer aloud before reading.*

# ELK / Observability in FastAPI — Interview Questions

> Format: 5 architectural questions with deep-dive answers, a multiple-choice
> knowledge check with an answer key, and a consolidated gotchas list.

---

## Part 1 — Architectural Deep-Dive Questions

### Q1. Why is `print(f"user {id} failed")` a production anti-pattern, and what replaces it?


**Deep dive.** Free-text logs are a dead end at scale. Elasticsearch indexes
*fields*, so structured JSON logs become a queryable, aggregatable dataset — you
can run `status:500 AND latency_ms:>1000 AND path:"/checkout"` and build
dashboards and alerts from it. `print`/f-string logs can only be grep'd line by
line, can't be aggregated (avg latency, error rate), and can't drive alerts. The
replacement is a JSON formatter emitting one object per line to stdout, with
consistent field names across all services, shipped to ELK by a forwarder. Also,
`print` bypasses log levels and handlers entirely — use the `logging` module.

---

### Q2. A request fails somewhere across five services. Walk me through finding where, and what makes it possible.


**Deep dive.** You need a **correlation (trace) ID**. Generate it at the edge — or
accept an inbound `X-Request-ID` from the gateway/upstream — propagate it through
every downstream call via headers, and stamp it on every log line and trace span.
In FastAPI, a middleware sets the ID into a `contextvar` at request start, and the
JSON formatter reads that contextvar so *every* line for the request carries it,
even across `await` points. Then you filter Kibana (or APM) by that single ID to
reconstruct the whole request timeline in order and pinpoint the failing hop.
Without correlation IDs, cross-service debugging in interleaved logs is guesswork.

---

### Q3. Compare logs, metrics, and traces. Which do you alert on?


**Deep dive.** They're complementary. **Metrics** are cheap numeric aggregates
(rates, latencies, counts) — they tell you *something is wrong* and are what you
**alert** on, using symptom-based golden signals (latency p99, traffic, errors,
saturation). **Traces** follow one request across services and tell you *where*
time went or which hop failed. **Logs** are detailed per-event records that tell
you *why* (the exception + context) once you've localized the problem. Alert on
metrics, not logs, for things like latency — a histogram gives percentiles
cheaply; then pivot to traces to localize and logs (by correlation ID) to
root-cause. Alerting on causes ("CPU 80%") creates noise; alert on user-facing
symptoms tied to runbooks.

---

### Q4. Your Elasticsearch cluster filled its disk overnight. What went wrong and how do you prevent recurrence?


**Deep dive.** Logs are unbounded time-series data; without lifecycle management,
indices grow until the disk is full — a predictable outage. Prevention is **Index
Lifecycle Management (ILM)**: roll indices over by age/size, transition older data
through hot → warm → cold tiers on cheaper storage, and delete beyond the
retention period. Also guard against **mapping explosions** — logging dynamic or
unbounded field names (e.g., serializing a whole object with arbitrary keys) bloats
the index mapping and can destabilize the cluster — and consider **sampling**
high-volume success logs while never sampling errors. Add capacity planning and a
disk-saturation alert so you catch it before it recurs.

---

### Q5. What are the security and PII considerations of logging, and how do you enforce them?


**Deep dive.** Logs are widely readable (whole org + tooling) and long-lived, so
they're a prime data-leak vector. Never log secrets (passwords, tokens, API keys,
full card numbers) or unnecessary PII. Enforcement is layered: a code-review
policy and lint rules; a structured-logging helper that only accepts an explicit
allow-list of fields (so you can't accidentally dump a whole request/user object);
redaction/masking filters in the logging pipeline (e.g., mask anything matching
token/password patterns) as a safety net; and retention limits via ILM so leaked
data doesn't live forever. Logging a password (as the junior endpoint does) is a
reportable security incident, not a style nit.

---

## Part 2 — Multiple-Choice Knowledge Check

**1. Structured JSON logging beats f-string logging in ELK primarily because:**
- A) it's shorter
- B) Elasticsearch can index fields, enabling search/aggregation/alerts
- C) it uses less disk
- D) it's required by Python

**2. To trace one request across many services you use a:**
- A) bigger log level
- B) correlation / trace ID propagated via headers and context
- C) separate log file per request
- D) database transaction

**3. You should alert primarily on:**
- A) raw CPU usage
- B) user-facing symptoms (latency, errors, saturation, traffic)
- C) log line count
- D) disk reads

**4. An Elasticsearch cluster running out of disk from logs is prevented by:**
- A) bigger log messages
- B) Index Lifecycle Management (rollover + tiering + deletion)
- C) disabling logging
- D) more Kibana dashboards

**5. Logging a user's password is:**
- A) fine if the log is internal
- B) a security incident — never log secrets/PII
- C) required for auditing
- D) acceptable at DEBUG level

### Answer Key
1. **B** — indexed fields make logs queryable and alertable.
2. **B** — a propagated correlation/trace ID stitches the request.
3. **B** — alert on symptoms (golden signals), not raw causes.
4. **B** — ILM caps unbounded time-series growth.
5. **B** — never log secrets; it's an incident.

---

## Part 3 — Gotchas Checklist

- **No `print` in production.** Use the `logging` module with a JSON formatter to
  stdout (12-factor); let the platform ship logs to ELK.
- **One JSON object per line** with **consistent field names** across services, or
  you can't aggregate across them.
- **Always attach a correlation ID** (set in middleware via a `contextvar` so it
  survives `await`); return it in a response header for client-side correlation.
- **Never log secrets or PII.** Use an allow-list logging helper + pipeline
  redaction as a safety net; logging a password is an incident.
- **Beware mapping explosions** — don't log unbounded/dynamic field names; keep
  fields bounded and typed.
- **Configure ILM** (rollover, hot/warm/cold, retention) or the cluster *will*
  fill its disk.
- **Alert on symptoms, not causes**, and make every alert actionable with a
  runbook — noisy alerts train people to ignore them.
- **Prefer metrics over logs for latency alerting** (histograms give cheap
  percentiles); use logs to investigate specific slow requests.
- **Exceptions belong in a field** with the stack trace, not smeared across
  multiple free-text lines.

---
# Database Scaling in FastAPI — Interview Questions

---
## 🏆 Interview Questions — Database Scaling in FastAPI — Interview Questions

*Model answers included. Say your answer aloud before reading.*

# Database Scaling in FastAPI — Interview Questions

> Format: 5 architectural questions with deep-dive answers, a multiple-choice
> knowledge check with an answer key, and a consolidated gotchas list.

---

## Part 1 — Architectural Deep-Dive Questions

### Q1. What is the N+1 query problem, why is it so common with ORMs, and how do you fix it?


**Deep dive.** N+1 is: one query to fetch a list of N rows, then one *additional*
query per row to fetch a related object — N+1 round trips total. It's rampant with
ORMs because lazy-loading a relationship (`order.customer`) looks like a simple
attribute access but silently fires a query each time, hidden inside a loop. At
1,000 rows that's 1,001 round trips, each paying network + query-planning
overhead — an endpoint that's fast on 5 rows and times out on 5,000. Fixes: eager
-load the relationship in one query (SQLAlchemy `selectinload`/`joinedload`), or
collect the distinct foreign keys and issue a single `IN (...)` batch, turning
O(n) round trips into O(1). Detection: log/count queries per request and alert on
outliers.

---

### Q2. Walk me up the database scaling ladder for a read-heavy app. Where does sharding fit?


**Deep dive.** Cheapest first, stop when the bottleneck is solved. (1) **Optimize**
— add indexes, kill N+1s, tune slow queries, size the connection pool; often
enough alone. (2) **Cache** hot reads (Redis) to offload the DB. (3) **Read
replicas** — route reads off the primary to scale reads horizontally. (4)
**Shard/partition** — only when *writes* or *storage* exceed a single primary,
because it's the most operationally complex step (cross-shard queries,
rebalancing). Sharding is last, not first; jumping to it prematurely adds huge
complexity for a problem an index or a cache would have solved.

---

### Q3. You route reads to a replica and users report "I saved it but it's not there." Explain and mitigate.


**Deep dive.** Replicas replicate asynchronously, so they lag the primary by
milliseconds to seconds. A user who writes to the primary then immediately reads
from a lagging replica sees stale data — the **read-your-writes** problem.
Mitigations: route reads that *must* see the latest write back to the **primary**
(e.g., right after the user's own write, or within a short sticky window); track
the write's log position and read from a replica only once it has caught up; or
accept staleness where the domain tolerates it (a public view count). The key
skill is classifying which reads need freshness and paying the primary-routing
cost only for those, rather than treating all reads identically.

---

### Q4. You need to update the DB and publish an event. How do you avoid losing the event if the process crashes between them?


**Deep dive.** This is the **dual-write problem**: the DB and the broker are two
systems, so a crash after the DB commit but before the publish loses the event and
leaves services inconsistent (publishing first has the mirror flaw). The
**Transactional Outbox** solves it: within a *single* DB transaction, write the
business row **and** an `outbox` row; they commit atomically and can't diverge. A
separate relay (or CDC/Debezium) reads unpublished outbox rows, publishes them to
the broker, and marks them sent. Because the relay can crash and re-send, delivery
is at-least-once, so consumers must be idempotent. This is the reliable bridge
from your database into an event stream.

---

### Q5. How do you choose a shard key, and what goes wrong with a bad one?


**Deep dive.** The shard key must (a) distribute load evenly to avoid hotspots and
(b) align with the dominant access pattern to avoid cross-shard queries. Bad keys:
sharding by `country` concentrates traffic on one shard; sharding by a
monotonically increasing timestamp/ID sends all new writes to the last shard (a
hotspot). A good key like `customer_id` spreads load and co-locates a customer's
data so common per-customer queries stay single-shard. Cross-shard queries
(joins/aggregations spanning shards) are slow and complex, and **rebalancing** when
adding a shard is painful — which is why consistent hashing is used to bound how
much data moves. The shard key is a near-irreversible decision; get it right up
front.

---

## Part 2 — Multiple-Choice Knowledge Check

**1. Fetching a list then lazily loading each row's relation in a loop causes:**
- A) a deadlock
- B) the N+1 query problem
- C) a cache stampede
- D) replication lag

**2. The fix for N+1 is to:**
- A) add more replicas
- B) eager-load / batch the related rows in one query
- C) increase the connection pool
- D) disable the ORM

**3. Reading immediately from an async replica after a write can show stale data — this is:**
- A) the dual-write problem
- B) read-your-writes / replication lag
- C) a cache miss
- D) an N+1 query

**4. Atomically saving a row and an event to publish later uses the:**
- A) Saga pattern
- B) Transactional Outbox pattern
- C) CQRS pattern
- D) Singleton pattern

**5. In the scaling ladder, sharding should be:**
- A) the first thing you try
- B) used only when writes/storage exceed a single primary
- C) applied to every table by default
- D) a replacement for indexes

### Answer Key
1. **B** — per-row lazy loads = N+1.
2. **B** — eager-load or batch into one query.
3. **B** — async replica lag causes read-your-writes staleness.
4. **B** — the outbox commits row + event in one transaction.
5. **B** — shard last, only when a single primary is exceeded.

---

## Part 3 — Gotchas Checklist

- **N+1 hides behind ORM lazy loading.** `for x in list: x.relation` silently
  fires a query per row — eager-load or batch, and count queries per request.
- **Not all reads can go to a replica.** Reads that must see a just-written value
  belong on the primary (read-your-writes); classify freshness needs.
- **Replication lag is real** — treat replicas as eventually consistent and design
  for it, don't assume instant sync.
- **Dual writes lose data on crash.** Use the Transactional Outbox to make the
  state change and its event atomic; consumers stay idempotent.
- **Shard last.** Optimize → cache → replicas → shard. Sharding adds cross-shard
  queries and rebalancing pain; don't reach for it first.
- **The shard key is near-irreversible** — pick one that avoids hotspots and keeps
  common queries single-shard.
- **Size the connection pool** — unbounded connections exhaust the DB; too few
  serialize requests. Pool once (see FastAPI `lifespan`).
- **Missing indexes** turn point lookups into full scans — the cheapest scaling
  win is usually an index, not more hardware.

---
# Event-Driven Systems in FastAPI — Interview Questions

---
## 🏆 Interview Questions — Event-Driven Systems in FastAPI — Interview Questions

*Model answers included. Say your answer aloud before reading.*

# Event-Driven Systems in FastAPI — Interview Questions

> Format: 5 architectural questions with deep-dive answers, a multiple-choice
> knowledge check with an answer key, and a consolidated gotchas list.

---

## Part 1 — Architectural Deep-Dive Questions

### Q1. An order endpoint synchronously calls inventory, email, and analytics. What's wrong and what does publishing an event fix?


**Deep dive.** Synchronous chaining creates **temporal and logical coupling**: the
endpoint must know every downstream, the client waits for all of them, and a slow
or failing *non-critical* step (email) makes the *critical* operation (placing the
order) slow or fail. Adding a new reaction means editing the tested endpoint.
Publishing an `OrderPlaced` **event** inverts this: the endpoint does only its own
critical work and emits a fact; independent consumers react asynchronously. Now
email being down doesn't fail the order (it retries / dead-letters), you add a
fraud-check consumer without touching the producer, and the client isn't blocked
on downstream work. The trade-off is eventual consistency and more moving parts.

---

### Q2. Your broker delivers at-least-once. What must every consumer guarantee, and how?


**Deep dive.** Every consumer must be **idempotent** — processing the same message
twice has the same effect as once. At-least-once means duplicates are inevitable
(producer retries, redeliveries, rebalances), so consumers dedupe on a unique
`event_id` (persisted in a DB table or a Redis set with TTL) and no-op on repeats,
or design the operation to be naturally idempotent (`set status = shipped` rather
than `balance += 1`). Combining at-least-once delivery with idempotent consumers
yields exactly-once *effect*, which is the achievable form of "exactly-once" —
brokers can't give you true end-to-end exactly-once for free. Idempotency is the
single most important correctness property in an event system.

---

### Q3. Compare choreography and orchestration. When do you pick each?


**Deep dive.** In **choreography**, services react to each other's events with no
central coordinator — maximal decoupling, great for simple few-step flows, but the
overall business process becomes emergent and hard to see or change as steps
multiply ("who does what?"). In **orchestration**, a central orchestrator directs
each step — the workflow lives in one place, easy to reason about and modify, at
the cost of a component coupled to all participants. Rule of thumb: choreography
between loosely-coupled bounded contexts; orchestration (often a **saga
orchestrator**) inside a complex, multi-step workflow that needs explicit control
and error handling. Many systems use both at different granularities.

---

### Q4. How do you implement a transaction that spans three services (no distributed 2PC)?


**Deep dive.** Use a **saga**: a sequence of local transactions, each publishing an
event that triggers the next step. If a step fails, you run **compensating
transactions** to undo the completed steps in reverse order (reserve inventory →
charge payment fails → *release* inventory). This achieves eventual consistency
without two-phase commit, which doesn't scale and creates cross-service locking.
Sagas come in two flavors — choreographed (each service listens and reacts) or
orchestrated (a coordinator drives steps and compensations). Compensations must be
idempotent and you must design for the awkward case where the *undo* itself fails
(alerting, manual intervention, retries).

---

### Q5. A single "poison" message keeps failing and blocks your consumer. What do you do, and how do you preserve ordering?


**Deep dive.** Retry a **bounded** number of times, then route the message to a
**Dead-Letter Queue** so the rest of the stream keeps flowing, and continue. The
DLQ preserves the message for inspection, fixing, and replay, and its depth should
be an alerting signal. Blocking the whole partition on one bad record turns a
single failure into an outage. For **ordering**: global ordering across a topic is
expensive and kills throughput; brokers like Kafka guarantee order only *within a
partition*, so you choose a partition key (e.g., `order_id`) that keeps all events
for one entity on the same partition (ordered) while different entities process in
parallel across partitions — the event-stream analogue of a shard key.

---

## Part 2 — Multiple-Choice Knowledge Check

**1. Synchronously calling every downstream service from an endpoint causes:**
- A) idempotency
- B) tight temporal/logical coupling and cascading failure
- C) eventual consistency
- D) horizontal scaling

**2. Because brokers deliver at-least-once, consumers must be:**
- A) synchronous
- B) idempotent
- C) stateless only
- D) single-threaded

**3. "Exactly-once effect" in practice is achieved by:**
- A) a magic broker setting
- B) at-least-once delivery + idempotent consumers
- C) at-most-once delivery
- D) two-phase commit

**4. A distributed transaction across services without 2PC uses the:**
- A) Outbox pattern
- B) Saga pattern with compensating transactions
- C) Singleton pattern
- D) Strategy pattern

**5. A repeatedly failing "poison" message should be:**
- A) retried forever
- B) sent to a dead-letter queue after bounded retries
- C) silently dropped
- D) logged with print()

### Answer Key
1. **B** — sync chaining couples services and cascades failure.
2. **B** — at-least-once ⇒ consumers must dedupe (idempotent).
3. **B** — at-least-once + idempotency = exactly-once effect.
4. **B** — sagas coordinate via compensations, no 2PC.
5. **B** — bounded retries then DLQ, keep the stream flowing.

---

## Part 3 — Gotchas Checklist

- **Don't chain services synchronously** for non-critical work — publish an event
  so a slow/failed consumer can't fail or slow the critical path.
- **Assume at-least-once delivery**; make every consumer **idempotent** (dedupe on
  `event_id` or use naturally idempotent operations).
- **Events are past-tense facts** (`OrderPlaced`), not commands (`PlaceOrder`) —
  facts keep consumers decoupled and independently evolvable.
- **The dual-write trap**: don't write the DB then publish in two steps — a crash
  between loses the event. Use the Transactional Outbox (see topic 07).
- **Handle poison messages** with bounded retries + a dead-letter queue, and alert
  on DLQ depth; never block the partition on one bad record.
- **Ordering is per-partition**, not global — pick a partition key (e.g.
  `order_id`) to keep an entity's events ordered while parallelizing others.
- **Compensations must be idempotent**, and you must plan for a failed undo.
- **Eventual consistency is the cost** of decoupling — make it explicit to
  stakeholders; it's not the right model for operations needing an immediate
  consistent answer.

---
# Concurrency & Parallelism — Interview Questions

---
## 🏆 Interview Questions — Concurrency & Parallelism — Interview Questions

*Model answers included. Say your answer aloud before reading.*

# Concurrency & Parallelism — Interview Questions

> Format: 5 architectural questions with deep-dive answers, a multiple-choice
> knowledge check with an answer key, and a consolidated gotchas list.

---

## Part 1 — Architectural Deep-Dive Questions

### Q1. What is the GIL, and does it mean threads are useless in Python?


**Deep dive.** The Global Interpreter Lock is a mutex in CPython that allows only
one thread to execute Python bytecode at a time. It exists to keep the
interpreter's own internals (notably reference counts) consistent without
fine-grained locking everywhere. Two consequences matter. First, it makes threads
**useless for CPU-bound** Python: a hot numeric loop holds the GIL, so extra
threads add scheduling overhead and give zero speedup — you need processes for
that. Second, it does **not** make your own operations atomic: `x += 1` is
read/add/write and the GIL can be released between steps, so threads still race
and still need locks. Threads remain genuinely useful for **I/O-bound** work,
because blocking calls (`socket.recv`, `time.sleep`, most C-level I/O) release the
GIL, letting other threads run while one waits. So the honest answer is: threads
overlap *waiting*, not *computing*.

---

### Q2. You have a task that's slow. How do you decide between threads, processes, and asyncio?


**Deep dive.** Classify the bottleneck first. If it's **I/O-bound** (waiting on
network, disk, or another service), the CPU is idle during the wait, so
concurrency helps: use **threads** for a modest number of blocking calls, or
**asyncio** when you need thousands of concurrent connections cheaply and can use
async-native libraries end to end. If it's **CPU-bound** (parsing, hashing,
number crunching in pure Python), threads and asyncio give *nothing* because of
the GIL — use **processes** (`ProcessPoolExecutor`) for true multi-core
parallelism, accepting the cost of pickling data across the boundary and higher
startup. The trade-off summary: threads are cheap and share memory but risk races
and don't scale CPU; processes scale CPU but don't share memory; asyncio scales
I/O fan-out enormously on one thread but poisons instantly if anything blocks the
loop. The senior answer names the workload shape, not a favorite tool.

---

### Q3. Reports say a counter is occasionally wrong under load, but it's fine in tests. How do you diagnose and fix it?


**Deep dive.** "Correct in tests, wrong under load, non-deterministic" is the
signature of a **race condition** on shared mutable state. The counter update is
almost certainly a non-atomic read-modify-write (`n += 1`): two threads read the
same value and one increment is lost, so the total can only ever *under*-count.
Tests miss it because they don't create the contended interleaving; to reproduce
deterministically, force a yield point between the read and the write (or crank up
thread count and iterations) and you'll see the drift. The fix is to make the
critical section **atomic** — wrap the read-modify-write in a `Lock` — or, better,
eliminate the shared state: have each worker accumulate locally and combine
results at the end, or route updates through a `queue.Queue`. Keep the locked
region minimal (guard the write, not any I/O) so you don't trade a race for a
contention or deadlock problem.

---

### Q4. Two threads deadlock intermittently transferring between accounts. What causes it and how do you prevent it?


**Deep dive.** Deadlock needs a **cycle in lock acquisition**: thread A locks
account 1 then waits for account 2, while thread B locks account 2 then waits for
account 1 — each holds what the other needs, forever. It's intermittent because it
only manifests when the two transfers overlap in that opposite order. The robust,
standard prevention is a **global lock ordering**: every thread acquires the two
locks in the same stable order (e.g. by the smaller account id), which makes a
cycle impossible by construction. Complementary tactics: hold locks for the
shortest possible span, never perform I/O or call out to other code while holding
a lock, use `acquire(timeout=...)` to fail loudly instead of hanging, and prefer
lock-free designs (a single queue, or an atomic conditional DB update) so there's
no second lock to order. "Just add more locks" makes deadlock *more* likely, not
less.

---

### Q5. Why is calling a blocking function inside an async coroutine catastrophic, and how do you handle unavoidable CPU or blocking work?


**Deep dive.** asyncio is **cooperative single-threaded** concurrency: one thread
runs all coroutines, and a coroutine only yields control when it `await`s. A
synchronous blocking call — `time.sleep`, a sync DB driver, a heavy CPU loop —
never yields, so it **freezes the entire event loop**: every other task, every
in-flight request, every timer stalls until it returns. In a service this shows up
as catastrophic tail-latency and dropped throughput under concurrency, even though
a single request looks fine. The fix is to keep the loop free: offload blocking
**I/O** with `asyncio.to_thread(...)` (or a thread pool) and offload **CPU-bound**
work to a `ProcessPoolExecutor` via `loop.run_in_executor`, then `await` the
result — the loop stays responsive while the work runs elsewhere. The deeper
discipline is to use async-native libraries throughout so blocking calls never
sneak onto the loop in the first place, and to bound every await with a timeout so
a slow dependency degrades gracefully instead of hanging.

---

## Part 2 — Multiple-Choice Knowledge Check

**1. The GIL guarantees that:**
- A) `counter += 1` is atomic across threads
- B) only one thread runs Python bytecode at a time
- C) threads speed up CPU-bound Python
- D) locks are never needed

**2. For CPU-bound pure-Python work, the right tool is:**
- A) more threads
- B) asyncio
- C) processes (a process pool)
- D) a bigger GIL

**3. `x += 1` across threads without a lock can lose updates because it is:**
- A) atomic
- B) a read-modify-write that can interleave
- C) protected by the GIL
- D) a syntax error

**4. The standard way to prevent deadlock between two locks is:**
- A) acquire them in a consistent global order
- B) add a third lock
- C) use more threads
- D) ignore it; it's rare

**5. Calling `time.sleep(5)` inside an `async def` coroutine will:**
- A) sleep only that coroutine
- B) block the entire event loop for 5 seconds
- C) raise an exception
- D) run on another core

**6. A bounded `queue.Queue(maxsize=N)` provides:**
- A) parallel CPU execution
- B) back-pressure so a fast producer can't exhaust memory
- C) deadlock immunity
- D) a way to bypass the GIL

### Answer Key
1. **B** — the GIL serializes bytecode; it does not make your operations atomic.
2. **C** — only processes achieve true multi-core parallelism for Python code.
3. **B** — read/add/write can interleave, so an increment is lost.
4. **A** — a stable global acquisition order makes a lock cycle impossible.
5. **B** — a sync sleep never yields, freezing every task on the loop.
6. **B** — `maxsize` blocks the producer, matching it to the consumer's rate.

---

## Part 3 — Gotchas Checklist

- **The GIL is not a lock for *your* data.** It protects interpreter internals,
  not your read-modify-write — `+=`, `list[i] = ...`, check-then-act all still race.
- **Threads don't speed up CPU work.** If a hot loop is the bottleneck, threads add
  overhead and no speedup; move to processes.
- **Never block the event loop.** No `time.sleep`, sync DB drivers, or CPU loops in
  a coroutine — offload with `to_thread` / a process pool and `await` the result.
- **Keep critical sections tiny.** Guard the write, not the network call; a lock
  held across I/O is a throughput cliff and a deadlock invitation.
- **Order your locks.** Always acquire multiple locks in the same global order to
  make deadlock structurally impossible.
- **Prefer a queue to shared state.** `queue.Queue` is internally synchronized;
  producer/consumer handoff needs no manual locks.
- **Always bound your waits.** An `await` (or a blocking call) with no timeout is a
  latent hang; use `asyncio.wait_for` / socket timeouts.
- **Bound your buffers for back-pressure.** An unbounded queue turns a slow
  consumer into an out-of-memory crash under load.
- **Process-pool workers must be picklable, top-level functions** (Windows uses
  `spawn`); closures and lambdas won't serialize across the boundary.
- **Distributed check-then-act is the same race, scaled up.** "Read stock, then
  decrement" oversells under concurrency — fix it with an atomic conditional
  update or a row lock, not an in-process `Lock`.

---
# Networking, Security & Testing — Interview Questions

---
## 🏆 Interview Questions — Networking, Security & Testing — Interview Questions

*Model answers included. Say your answer aloud before reading.*

# Networking, Security & Testing — Interview Questions

> Format: 5 architectural questions with deep-dive answers, a multiple-choice
> knowledge check with an answer key, and a consolidated gotchas list.

---

## Part 1 — Architectural Deep-Dive Questions

### Q1. Why must you never store plaintext passwords, and why is a salted, slow KDF (bcrypt/PBKDF2/argon2) required rather than SHA-256?


**Deep dive.** A credential store *will* eventually leak, so the design goal is
to make a leaked table useless. Plaintext is game over. A **fast** hash like
SHA-256 is barely better: it's built for speed, so a GPU tries billions of
guesses per second, and without salting an attacker uses precomputed **rainbow
tables** and instantly sees which users share a password. Two properties fix
this. A per-user random **salt** makes identical passwords hash differently and
defeats precomputation — every table must be attacked from scratch. A
deliberately **slow** key-derivation function (PBKDF2 with many rounds, or
memory-hard bcrypt/scrypt/argon2) makes *each individual guess* expensive, so
brute force becomes economically infeasible. Complete it with a **constant-time
compare** on verify (`hmac.compare_digest`) so response timing can't leak the
stored hash byte by byte. This is OWASP A02 (Cryptographic Failures).

---

### Q2. Explain JWT integrity vs confidentiality, and the trade-off between stateless tokens and server-side sessions.


**Deep dive.** A JWT is `header.payload.signature`, where the signature is an
HMAC (or asymmetric signature) over the first two parts under a server secret.
It provides **integrity** — you can detect that nobody altered the claims — but
**not confidentiality**: the payload is base64url-*encoded*, readable by anyone
holding the token, so it must never contain secrets. The verifier must
**recompute and compare the signature (in constant time) before trusting any
claim**; an unverified payload is attacker input. The stateless trade-off:
a JWT needs no session-store lookup per request (cheap horizontal scaling,
easy cross-service trust), but it **can't be revoked** before `exp`. So you keep
access tokens short-lived and add a rotating **refresh token**. A server-side
**session** is the opposite: instantly revocable and small on the wire, but every
request pays a store lookup and you must share/replicate that store across nodes.
Tokens favor scale and statelessness; sessions favor control and revocation.

---

### Q3. Precisely how does a parameterized query stop SQL injection, and what should back it up?


**Deep dive.** Injection happens because untrusted input is concatenated into a
query *string*, so the input can change the query's *structure* — `' OR '1'='1`
turns `WHERE name = '<input>'` into a tautology that returns every row (an auth
bypass), and `'; DROP TABLE ...` smuggles a second statement. A **parameterized
(prepared) query** sends the SQL text and the parameter values to the driver on
**separate channels**: the database parses the SQL *first* with placeholders, then
binds the values as opaque data that is *never re-parsed as SQL*. So `' OR '1'='1`
becomes a literal (non-existent) username, not code. This is structural, not
sanitization — you are not trying to escape dangerous characters, you are making
data and code physically distinct. Back it with **least privilege** (the app's DB
role can't `DROP` or read tables it doesn't need) so any missed query isn't
catastrophic, and with input validation at the boundary for defense in depth.

---

### Q4. Authentication vs authorization: what's the difference, and where do you enforce each in a microservice mesh?


**Deep dive.** **Authentication** establishes *who* the caller is (credentials →
identity, e.g. verify a token's signature and expiry). **Authorization**
establishes *what* that identity may do (identity → permission, e.g. a role or
policy check). They are separate: a genuine, unexpired `user` token must still be
refused an `admin` action — treating "valid token ⇒ allowed" is OWASP A01
(Broken Access Control). In a mesh, do **authentication at the edge** (the API
gateway validates the token once and forwards a trusted, signed identity — plus
**mTLS** so services authenticate *each other*), but keep **authorization close to
the resource**: each service enforces its own permissions on each action, because
only it knows what its data means. Never rely solely on a perimeter check — an
internal caller (or a bug) that reaches a service directly must still be
authorized. Enforce on the server, from a signed claim, per action.

---

### Q5. What does TLS provide, what does mTLS add, and what are the trade-offs?


**Deep dive.** TLS gives three guarantees on the wire: **confidentiality**
(eavesdroppers see only ciphertext), **integrity** (tampering in transit is
detected), and **server authentication** (the certificate chain proves you
reached the real host, not a man-in-the-middle). Standard TLS authenticates only
the *server*; the client stays anonymous at the transport layer (you authenticate
it separately, e.g. with a token). **Mutual TLS** adds a **client certificate**,
so *both* ends cryptographically prove identity. That's why service meshes use
mTLS for east-west traffic: every service-to-service call is mutually
authenticated and encrypted transparently, with no app code involved, giving you
zero-trust networking between services. The trade-offs: certificate **issuance,
distribution, and rotation** are real operational cost (a mesh/CA like SPIFFE or
Istio automates it), there's a modest handshake/CPU overhead, and short-lived
certs need reliable renewal or you cause outages. TLS is table stakes for any
public traffic; mTLS is the standard for internal zero-trust.

---

## Part 2 — Multiple-Choice Knowledge Check

**1. The primary reason to use bcrypt/PBKDF2/argon2 instead of a single SHA-256 for passwords is:**
- A) SHA-256 output is too short
- B) they are deliberately slow and salted, making brute force infeasible
- C) SHA-256 is not cryptographically secure
- D) they encrypt the password so it can be decrypted later

**2. The signature on a JWT proves:**
- A) the payload is encrypted and secret
- B) the claims have not been tampered with (integrity)
- C) the token cannot expire
- D) the user is authorized for admin actions

**3. Parameterized queries prevent SQL injection because:**
- A) they escape every quote character in the input
- B) they send SQL and data on separate channels so input is never parsed as code
- C) they run inside a database transaction
- D) they hash the user input first

**4. A valid, unexpired token from a `user` account requesting an admin-only action should be:**
- A) allowed — the token is valid
- B) refused — authorization is a separate check from authentication
- C) allowed if the token has a `role` field of any value
- D) refused only if the token is expired

**5. Compared with a mock, a fake (a working in-memory implementation) is usually preferred because:**
- A) it makes tests run on the network
- B) it asserts on interactions, coupling tests to implementation
- C) it lets you assert on observable state and survives refactors
- D) it removes the need for any assertions

**6. mTLS differs from ordinary TLS in that it additionally provides:**
- A) confidentiality of the payload
- B) client authentication via a client certificate
- C) faster handshakes
- D) protection against SQL injection

### Answer Key
1. **B** — salt defeats precomputation; slowness makes each guess costly.
2. **B** — HMAC/signature proves integrity, not secrecy (payload is only encoded).
3. **B** — separating SQL from data makes input structurally incapable of being code.
4. **B** — authentication ≠ authorization; enforce the role check server-side.
5. **C** — fakes verify behavior via state, so refactors don't break them.
6. **B** — mutual TLS authenticates the client too, not just the server.

---

## Part 3 — Gotchas Checklist

- **Never store plaintext or a fast/unsalted hash.** Salt per user + a slow KDF
  (bcrypt/argon2/PBKDF2) + constant-time compare (`hmac.compare_digest`).
- **A JWT payload is encoded, not encrypted.** Anyone with the token can read it;
  put no secrets in it, and **verify the signature before trusting any claim**.
- **Short token lifetimes + rotation.** A stateless JWT can't be revoked mid-life;
  bound the damage with `exp` and refresh-token rotation.
- **Injection is structural, not a sanitization problem.** Use parameterized
  queries; back them with least-privilege DB roles.
- **Authn ≠ authz.** A valid token is not permission — enforce role/policy checks
  per action, on the server, from a signed claim.
- **Enforce authorization close to the resource,** not only at the gateway; an
  internal caller must still be authorized.
- **Don't log secrets.** Redact passwords, tokens, and `Authorization` headers in
  structured logs and audit trails.
- **Prefer fakes to mocks.** Assert on observable state, not on how a collaborator
  was called, so refactors don't break green tests.
- **Shape the pyramid.** Many unit, some integration, few e2e; slow, flaky e2e
  suites hide real failures. Seed randomness/time to kill flakiness.
- **mTLS for east-west traffic.** Mutual certs authenticate both ends; budget for
  certificate rotation (let a mesh/CA automate it).

---
## Scenario-Based Code Questions -- All Modules

---
## Scenario-Based Code Questions -- Python Foundations

### Scenario 1 -- Retry Decorator with Exponential Backoff + Jitter

**Context:** BuildFast's GitHub API client gets 429 rate-limit errors.
Implement `@retry(times, exceptions, base_delay, cap)` with exponential backoff + random jitter to prevent thundering-herd.

**Requirements:**
- Delay = `min(base * 2^attempt + random(0, 0.5), cap)` seconds
- Raises the LAST exception if all retries fail
- Preserves `__name__` / `__doc__` via `functools.wraps`

**Try it, then scroll to the solution.**

In [ ]:
# -- ATTEMPT (scaffold) --
import functools, time, random

def retry(times=3, exceptions=(Exception,), base_delay=1.0, cap=60.0):
    # YOUR CODE HERE
    pass

# Quick test
call_count = 0

@retry(times=3, exceptions=(ValueError,), base_delay=0.001)
def flaky():
    global call_count; call_count += 1
    if call_count < 3: raise ValueError('transient')
    return 'ok'

print(flaky(), call_count)  # should print: ok  3

In [ ]:
# -- SOLUTION --
import functools, time, random

def retry(times=3, exceptions=(Exception,), base_delay=1.0, cap=60.0):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            last = None
            for attempt in range(times):
                try:
                    return fn(*args, **kwargs)
                except exceptions as e:
                    last = e
                    if attempt < times - 1:
                        delay = min(base_delay * (2**attempt) + random.uniform(0, 0.5), cap)
                        time.sleep(delay)
            raise last
        return wrapper
    return decorator

# -- Verify --
call_count = 0

@retry(times=3, exceptions=(ValueError,), base_delay=0.001)
def flaky():
    global call_count; call_count += 1
    if call_count < 3: raise ValueError('transient')
    return 'ok'

assert flaky() == 'ok' and call_count == 3
assert flaky.__name__ == 'flaky'
print(f'Result: ok after {call_count} attempts, __name__ preserved: {flaky.__name__!r}')

**Analysis:** 3-level nesting (factory -> decorator -> wrapper). `min(..., cap)` prevents unbounded waits. Jitter desynchronises concurrent callers. `functools.wraps` preserves identity for FastAPI/Pydantic route introspection.

### Scenario 2 -- Lazy CSV Pipeline (Generator Chain)

**Context:** BuildFast exports 10M+ build records. Loading into a list exhausts 16GB RAM. Stream one row at a time using generators.

**Requirements:** `read_csv` -> `filter_rows` -> `transform` -> `to_csv_lines` -- each step is lazy. Peak memory = O(1).

In [ ]:
# -- SOLUTION --
import csv, io, sys

def read_csv(fileobj):
    for row in csv.DictReader(fileobj): yield dict(row)

def filter_rows(rows, key, value):
    for row in rows:
        if row.get(key) == value: yield row

def transform(rows, fn):
    for row in rows: yield fn(row)

def to_csv_lines(rows):
    for row in rows: yield ','.join(str(v) for v in row.values())

# -- Test --
data = 'id,status,dur_ms\n1,failed,8900\n2,success,1200\n3,failed,12000'
pipeline = to_csv_lines(
    transform(
        filter_rows(read_csv(io.StringIO(data)), 'status', 'failed'),
        lambda r: {**r, 'dur_s': float(r['dur_ms'])/1000}
    )
)
for line in pipeline: print(line)
gen = to_csv_lines(filter_rows(read_csv(io.StringIO(data)), 'status', 'failed'))
print(f'Generator size: {sys.getsizeof(gen)} bytes -- constant regardless of file size')

### Scenario 3 -- Descriptor-Based Field Validator

**Context:** BuildFast pipeline config fields need type + range validation on EVERY assignment (not just `__init__`). Build a reusable descriptor.

```python
class PipelineConfig:
    max_parallel = TypedField(int, min_val=1, max_val=50)
    name         = TypedField(str)
cfg.max_parallel = -1  # ValueError!
```

In [ ]:
# -- SOLUTION --
class TypedField:
    def __init__(self, ftype, min_val=None, max_val=None):
        self.ftype, self.min_val, self.max_val = ftype, min_val, max_val

    def __set_name__(self, owner, name): self._attr = f'_{name}'

    def __get__(self, obj, t=None): return self if obj is None else getattr(obj, self._attr, None)

    def __set__(self, obj, value):
        if not isinstance(value, self.ftype):
            raise TypeError(f'{self._attr[1:]}: expected {self.ftype.__name__}, got {type(value).__name__}')
        if self.min_val is not None and value < self.min_val:
            raise ValueError(f'{self._attr[1:]} >= {self.min_val} required, got {value}')
        if self.max_val is not None and value > self.max_val:
            raise ValueError(f'{self._attr[1:]} <= {self.max_val} required, got {value}')
        setattr(obj, self._attr, value)

class PipelineConfig:
    max_parallel    = TypedField(int, min_val=1, max_val=50)
    timeout_minutes = TypedField(int, min_val=1, max_val=1440)
    name            = TypedField(str)

cfg = PipelineConfig()
cfg.name = 'frontend-ci'; cfg.max_parallel = 4
for bad, exc in [(-1, ValueError), (51, ValueError), ('5', TypeError)]:
    try: cfg.max_parallel = bad
    except (ValueError, TypeError) as e: print(f'  Rejected {bad!r}: {e}')
print('Descriptor validates on every assignment (not just __init__) |/')

### Scenario 4 -- Plugin Registry via __init_subclass__

**Context:** BuildFast report exporters self-register by subclassing `Exporter`. No manual registry call needed.

```python
class JsonExporter(Exporter, fmt='json'):
    def export(self, data): return json.dumps(data)
Exporter.create('json').export({'a': 1})  # works immediately
```

In [ ]:
# -- SOLUTION --
import json as _json
from typing import ClassVar

class Exporter:
    _registry: ClassVar[dict] = {}

    def __init_subclass__(cls, fmt: str = '', **kwargs):
        super().__init_subclass__(**kwargs)
        Exporter._registry[fmt or cls.__name__.lower()] = cls

    @classmethod
    def create(cls, fmt: str):
        if fmt not in cls._registry:
            raise KeyError(f'Unknown format {fmt!r}. Available: {list(cls._registry)}')
        return cls._registry[fmt]()

    def export(self, data) -> str: raise NotImplementedError

class JsonExporter(Exporter, fmt='json'):
    def export(self, data) -> str: return _json.dumps(data)

class CsvExporter(Exporter, fmt='csv'):
    def export(self, data) -> str: return ','.join(str(v) for v in data)

print('Registry:', list(Exporter._registry))
print(Exporter.create('json').export({'build': 'success'}))
print(Exporter.create('csv').export([1, 2, 3]))
try: Exporter.create('xml')
except KeyError as e: print(f'Unknown: {e}')

---
## Scenario-Based Code Questions -- DSA

### Scenario 1 -- LRU Cache (O(1) get and put)

**Context:** ShopFlow caches rendered product HTML. On capacity, evict least-recently-used. Implement with O(1) for both operations using a doubly-linked list + hash map. **Do NOT use OrderedDict.**

```
cache = LRUCache(2)
cache.put(1, 'a'); cache.put(2, 'b')
cache.get(1)        # 'a' -- now MRU
cache.put(3, 'c')   # evicts key 2
cache.get(2)        # -1 (evicted)
```

In [ ]:
# -- SOLUTION --
class LRUCache:
    class Node:
        __slots__ = ('key', 'val', 'prev', 'next')
        def __init__(self, key=0, val=0):
            self.key, self.val, self.prev, self.next = key, val, None, None

    def __init__(self, capacity):
        self.cap, self.map = capacity, {}
        self.head, self.tail = self.Node(), self.Node()
        self.head.next, self.tail.prev = self.tail, self.head

    def _remove(self, n): n.prev.next, n.next.prev = n.next, n.prev

    def _push_front(self, n):
        n.prev, n.next = self.head, self.head.next
        self.head.next.prev = n; self.head.next = n

    def get(self, key):
        if key not in self.map: return -1
        n = self.map[key]; self._remove(n); self._push_front(n)
        return n.val

    def put(self, key, val):
        if key in self.map: self._remove(self.map[key])
        n = self.Node(key, val); self.map[key] = n; self._push_front(n)
        if len(self.map) > self.cap:
            lru = self.tail.prev; self._remove(lru); del self.map[lru.key]

cache = LRUCache(2)
cache.put(1, 'product_a'); cache.put(2, 'product_b')
assert cache.get(1) == 'product_a'
cache.put(3, 'product_c')
assert cache.get(2) == -1
assert cache.get(3) == 'product_c'
print('LRU Cache: all assertions passed |/')

### Scenario 2 -- Sliding Window Rate Limiter

**Context:** BuildFast limits each user to N requests per T-second sliding window. Unlike fixed windows, a sliding window counts requests in the ACTUAL last T seconds.

```
rl = SlidingWindowRateLimiter(3, 60)
rl.allow('u1', 0)   # True
rl.allow('u1', 65)  # True (t=0 expired)
```

In [ ]:
# -- SOLUTION --
from collections import defaultdict, deque

class SlidingWindowRateLimiter:
    def __init__(self, max_requests, window_seconds):
        self.max, self.window = max_requests, window_seconds
        self._log: dict = defaultdict(deque)

    def allow(self, user_id: str, timestamp: float) -> bool:
        dq = self._log[user_id]
        cutoff = timestamp - self.window
        while dq and dq[0] <= cutoff: dq.popleft()
        if len(dq) < self.max:
            dq.append(timestamp); return True
        return False

rl = SlidingWindowRateLimiter(3, 60)
results = [rl.allow('u1', t) for t in [0, 20, 40, 50, 65]]
print('allow() results:', results)
assert results == [True, True, True, False, True]
print('Sliding window rate limiter correct |/')

### Scenario 3 -- Pipeline Topological Sort + Cycle Detection

**Context:** BuildFast CI/CD jobs have dependencies. Detect circular dependencies and return valid execution order. Use Kahn's BFS (O(V+E)).

In [ ]:
# -- SOLUTION --
from collections import defaultdict, deque

def validate_pipeline(jobs: dict):
    in_deg = {j: 0 for j in jobs}
    for job, deps in jobs.items():
        for dep in deps:
            in_deg[job] += 1
            if dep not in in_deg: in_deg[dep] = 0
    queue = deque(j for j, d in in_deg.items() if d == 0)
    order = []
    while queue:
        job = queue.popleft(); order.append(job)
        for nj, deps in jobs.items():
            if job in deps:
                in_deg[nj] -= 1
                if in_deg[nj] == 0: queue.append(nj)
    return order if len(order) == len(in_deg) else None

jobs = {'checkout':[],'install':['checkout'],'test':['install'],'build':['test'],'deploy':['build']}
print('Valid order:', validate_pipeline(jobs))
cyclic = {'a':['c'],'b':['a'],'c':['b']}
print('Cyclic pipeline:', validate_pipeline(cyclic))  # None

### Scenario 4 -- Consistent Hashing Ring

**Context:** ShopFlow distributes product cache across 3 nodes. When a node is removed, only ~1/3 of keys should be remapped (not all).

In [ ]:
# -- SOLUTION --
import hashlib, bisect

class ConsistentHashRing:
    def __init__(self, replicas=100):
        self.replicas, self._ring, self._nodes = replicas, [], {}

    def _hash(self, key): return int(hashlib.md5(key.encode()).hexdigest(), 16)

    def add_node(self, node):
        for i in range(self.replicas):
            h = self._hash(f'{node}#{i}')
            self._nodes[h] = node; bisect.insort(self._ring, h)

    def remove_node(self, node):
        for i in range(self.replicas):
            h = self._hash(f'{node}#{i}')
            del self._nodes[h]; self._ring.pop(bisect.bisect_left(self._ring, h))

    def get_node(self, key):
        if not self._ring: return None
        idx = bisect.bisect_right(self._ring, self._hash(key)) % len(self._ring)
        return self._nodes[self._ring[idx]]

ring = ConsistentHashRing(50)
for n in ['cache-1','cache-2','cache-3']: ring.add_node(n)
keys = [f'user:{i}' for i in range(1000)]
before = {k: ring.get_node(k) for k in keys}
ring.remove_node('cache-2')
after = {k: ring.get_node(k) for k in keys}
remapped = sum(1 for k in keys if before[k] != after[k])
print(f'Remapped: {remapped}/1000 ({remapped/10:.1f}%) -- expect ~33%')

---
## Scenario-Based Code Questions -- System Design

### Scenario 1 -- Token Bucket Rate Limiter

**Context:** ShopFlow's API gateway rate-limits to `max_tokens` requests/sec using token bucket (allows short bursts).

**Critical:** Use `time.monotonic()` NOT `time.time()` -- immune to NTP jumps!

In [ ]:
# -- SOLUTION --
import time, threading
from collections import defaultdict

class TokenBucket:
    def __init__(self, max_tokens: float, refill_rate: float):
        self.max, self.rate = max_tokens, refill_rate
        self._tokens: dict = defaultdict(lambda: max_tokens)
        self._last:   dict = defaultdict(lambda: 0.0)   # 0.0 default; time.monotonic is falsy-unsafe
        self._lock   = threading.Lock()

    def consume(self, user_id: str, n: float = 1, now: float | None = None) -> bool:
        now = now if now is not None else time.monotonic()
        with self._lock:
            elapsed = max(0.0, now - self._last[user_id])
            self._tokens[user_id] = min(self.max, self._tokens[user_id] + elapsed * self.rate)
            self._last[user_id]   = now
            if self._tokens[user_id] >= n:
                self._tokens[user_id] -= n; return True
            return False

tb = TokenBucket(max_tokens=10, refill_rate=5)
assert tb.consume('u1', 5, now=0.0)           # 10 → 5 remaining
assert not tb.consume('u1', 6, now=0.0)        # only 5 left, need 6 → denied
assert tb.consume('u1', 10, now=1.0)           # +5 refilled → 10; consume all 10
print('Token bucket assertions passed ✓')


### Scenario 2 -- Bloom Filter

**Context:** BuildFast deduplicates build events before querying the DB. False positives OK (will re-check DB), false negatives NOT OK.

In [ ]:
# -- SOLUTION --
import math, hashlib

class BloomFilter:
    def __init__(self, capacity: int, error_rate: float = 0.01):
        self.size   = int(-capacity * math.log(error_rate) / (math.log(2)**2))
        self.hashes = int(self.size / capacity * math.log(2))
        self._bits  = bytearray(self.size)

    def _pos(self, item):
        return [int(hashlib.sha256(f'{i}:{item}'.encode()).hexdigest(), 16) % self.size
                for i in range(self.hashes)]

    def add(self, item: str):
        for p in self._pos(item): self._bits[p] = 1

    def might_contain(self, item: str) -> bool:
        return all(self._bits[p] for p in self._pos(item))

bf = BloomFilter(capacity=10_000, error_rate=0.01)
for i in range(1000): bf.add(f'event_{i}')
assert bf.might_contain('event_0') and bf.might_contain('event_999')
assert not bf.might_contain('event_99999')
fps = sum(1 for i in range(10000, 20000) if bf.might_contain(f'event_{i}'))
print(f'False positive rate: {fps/10000:.2%} (target: ~1%)')

### Scenario 3 — Thread-Safe Connection Pool

**Context:** ShopFlow's order service opens a new DB connection per request. Under 500 concurrent users the DB crashes. Implement a thread-safe `ConnectionPool` with `acquire()` as a context manager. Block when all connections are in use; replace broken connections instead of returning them to the pool.


In [ ]:
# -- SOLUTION --
import threading, queue, time
from contextlib import contextmanager

class FakeConn2:
    _n = 0
    def __init__(self): FakeConn2._n += 1; self.id = FakeConn2._n; self.broken = False
    def is_alive(self): return not self.broken
    def execute(self, sql): return f"conn#{self.id}: {sql}"

class ConnectionPool:
    def __init__(self, factory, size=10, timeout=5.0):
        self._factory = factory; self._timeout = timeout
        self._pool = queue.Queue(maxsize=size)
        for _ in range(size): self._pool.put(factory())

    @contextmanager
    def acquire(self):
        try: conn = self._pool.get(timeout=self._timeout)
        except queue.Empty: raise TimeoutError("No connection available")
        ok = True
        try:
            if not conn.is_alive(): conn = self._factory()
            yield conn
        except Exception: ok = False; conn.broken = True; raise
        finally: self._pool.put(conn if ok else self._factory())

pool = ConnectionPool(FakeConn2, size=3, timeout=1.0)
with pool.acquire() as c: print(c.execute("SELECT 1"))
results = []; threads = [threading.Thread(target=lambda: results.append(
    pool.acquire().__enter__().execute("SELECT concurrent"))) for _ in range(3)]
# simpler concurrent test
def task():
    with pool.acquire() as c: results.append(c.execute("work"))
ts = [threading.Thread(target=task) for _ in range(3)]
for t in ts: t.start()
for t in ts: t.join()
assert len(results) == 3
assert pool._pool.qsize() == 3
print("Connection pool ✓", results)


### Scenario 4 — Saga Orchestrator with Compensation

**Context:** ShopFlow checkout: Reserve inventory → Charge payment → Create order → Trigger fulfillment. If step N fails, steps 1…N-1 must be compensated in reverse. Implement a `SagaOrchestrator`; compensations must run even if a previous compensation raises.


In [ ]:
# -- SOLUTION --
from dataclasses import dataclass, field
from typing import Callable, Any

@dataclass
class SagaStep2:
    name: str; action: Callable; compensation: Callable

@dataclass
class SagaResult2:
    success: bool; completed: list = field(default_factory=list)
    failed: str | None = None; compensated: list = field(default_factory=list)

class SagaOrchestrator2:
    def execute(self, steps: list[SagaStep2]) -> SagaResult2:
        r = SagaResult2(success=False); done = []
        for step in steps:
            try: step.action(); done.append(step); r.completed.append(step.name)
            except Exception as exc:
                r.failed = step.name
                for s in reversed(done):
                    try: s.compensation(); r.compensated.append(s.name)
                    except Exception: pass
                return r
        r.success = True; return r

log2 = []
def mk(name, fail=False):
    return SagaStep2(name,
        lambda n=name, f=fail: (_ for _ in ()).throw(RuntimeError(n)) if f else log2.append(f"✓ {n}"),
        lambda n=name: log2.append(f"↩ {n}"))

# Happy path
orch = SagaOrchestrator2()
r = orch.execute([mk("reserve"), mk("charge"), mk("order"), mk("fulfil")])
assert r.success; print("Happy:", log2)

# Payment fails → compensate inventory only
log2.clear()
r = orch.execute([mk("reserve"), mk("charge", fail=True), mk("order")])
assert not r.success and r.failed == "charge"
assert r.compensated == ["reserve"]; print("Payment fail:", log2)

# Order fails → compensate charge + inventory
log2.clear()
r = orch.execute([mk("reserve"), mk("charge"), mk("order", fail=True)])
assert r.compensated == ["charge", "reserve"]; print("Order fail:", log2)
print("Saga orchestrator ✓")


### Scenario 5 — Dead Letter Queue with Retry + Replay

**Context:** BuildFast's build-event consumer retries up to `max_retries` times with exponential backoff (use a `sleep_fn` hook for testing). After exhausting retries, the message moves to a DLQ with the failure reason. The DLQ must support replaying messages back to the main queue (resetting attempt count).


In [ ]:
# -- SOLUTION --
import time
from collections import deque
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class Msg: id: str; payload: dict; attempt: int = 0

@dataclass
class DLQEntry2: message: Msg; reason: str; attempts: int

class MQ:
    def __init__(self): self._q: deque[Msg] = deque()
    def publish(self, m): self._q.append(m)
    def poll(self): return self._q.popleft() if self._q else None
    def __len__(self): return len(self._q)

class DLQ:
    def __init__(self): self._e: list[DLQEntry2] = []
    def park(self, m, reason): self._e.append(DLQEntry2(m, reason, m.attempt))
    def replay_all(self, target):
        while self._e: e = self._e.pop(0); e.message.attempt = 0; target.publish(e.message)
    def __len__(self): return len(self._e)

class Consumer2:
    def __init__(self, q, dlq, handler, max_retries=3, base_delay=1.0, sleep_fn=time.sleep):
        self._q=q; self._dlq=dlq; self._handler=handler
        self._max=max_retries; self._delay=base_delay; self._sleep=sleep_fn
        self.processed=[]
    def process_one(self):
        m = self._q.poll()
        if m is None: return False
        while m.attempt <= self._max:
            try: self._handler(m); self.processed.append(m.id); return True
            except Exception as exc:
                m.attempt += 1
                if m.attempt > self._max: self._dlq.park(m, str(exc)); return True
                self._sleep(self._delay * 2**(m.attempt-1))
        return True

# Test
sleeps = []
q, dlq = MQ(), DLQ()
q.publish(Msg("e1", {}))
Consumer2(q, dlq, lambda m: (_ for _ in ()).throw(RuntimeError("down")),
          max_retries=2, base_delay=0.001, sleep_fn=lambda d: sleeps.append(d)).process_one()
assert len(dlq)==1 and dlq._e[0].attempts==3
print(f"DLQ entry: attempts={dlq._e[0].attempts}, reason={dlq._e[0].reason!r}")
dlq.replay_all(q); assert len(dlq)==0; replayed=q.poll()
assert replayed.id=="e1" and replayed.attempt==0
print(f"Replay: id={replayed.id}, attempt={replayed.attempt}")
print("DLQ ✓")


### Scenario 6 — Service Registry with TTL-Based Health (Service Discovery)

**Context:** ShopFlow's API gateway must route to a healthy instance of each service. Implement a `ServiceRegistry` where instances register with a TTL, renew via heartbeat, and auto-expire. `get_instance(service)` returns a healthy instance using round-robin (or raises `LookupError`).


In [ ]:
# -- SOLUTION --
import threading
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class SvcInst:
    service: str; instance_id: str; host: str; port: int; ttl: float
    _last_hb: float = 0.0
    def is_alive(self, now): return (now - self._last_hb) < self.ttl  # 0.0 default is falsy-safe

class SvcRegistry:
    def __init__(self, clock: Callable[[], float]):
        self._c = clock; self._i: dict = defaultdict(dict)
        self._lock = threading.Lock(); self._rr: dict = defaultdict(int)

    def register(self, svc, iid, host, port, ttl=30.0):
        t = self._c()
        inst = SvcInst(svc, iid, host, port, ttl, _last_hb=t)
        with self._lock: self._i[svc][iid] = inst

    def heartbeat(self, svc, iid):
        with self._lock:
            if iid in self._i[svc]: self._i[svc][iid]._last_hb = self._c()

    def cleanup(self):
        now = self._c()
        with self._lock:
            for s in list(self._i):
                for iid in [k for k,v in self._i[s].items() if not v.is_alive(now)]:
                    del self._i[s][iid]

    def get_instance(self, svc) -> SvcInst:
        self.cleanup()
        with self._lock: alive = list(self._i[svc].values())
        if not alive: raise LookupError(f"No healthy instances: {svc!r}")
        idx = self._rr[svc] % len(alive); self._rr[svc] += 1; return alive[idx]

# Test with injectable clock
_t = [0.0]
reg = SvcRegistry(clock=lambda: _t[0])
for i in range(3): reg.register("svc", f"i{i}", "10.0.0.1", 8000+i, ttl=30.0)
assert len(list(reg._i["svc"].values())) == 3
seen = {reg.get_instance("svc").instance_id for _ in range(6)}
assert len(seen) == 3; print("Round-robin:", seen)
_t[0] = 31  # advance past TTL
reg._i["svc"]["i1"]._last_hb = 31; reg._i["svc"]["i2"]._last_hb = 31  # renew 2 only
alive = [v for v in reg._i["svc"].values() if v.is_alive(31)]
reg.cleanup(); assert "i0" not in reg._i["svc"]; print("Expired i0; alive:", list(reg._i["svc"].keys()))
try: reg.get_instance("missing"); assert False
except LookupError as e: print(f"LookupError ✓: {e}")
print("Service registry ✓")


### Scenario 7 — Backpressure Queue (Drop / Block strategies)

**Context:** ShopFlow analytics receives events 10× faster than the processor handles at flash-sale peaks. Implement a `BackpressureQueue` supporting `"block"` (caller blocks until space) and `"drop"` (reject immediately at capacity). Include `metrics()` returning `accepted`, `dropped`, `processed`. Thread-safe.


In [ ]:
# -- SOLUTION --
import queue, threading
from typing import Any, Literal

class BackpressureQueue2:
    def __init__(self, maxsize: int, strategy: Literal["block","drop"]="drop"):
        self._q = queue.Queue(maxsize=maxsize); self._s = strategy
        self._a = self._d = self._p = 0; self._lock = threading.Lock()

    def put(self, item: Any) -> bool:
        if self._s == "drop":
            try: self._q.put_nowait(item)
            except queue.Full:
                with self._lock: self._d += 1; return False
            with self._lock: self._a += 1; return True
        else:
            self._q.put(item)  # blocks
            with self._lock: self._a += 1; return True

    def get(self, timeout=None):
        item = self._q.get(timeout=timeout)
        with self._lock: self._p += 1; return item

    def metrics(self):
        with self._lock: return {"accepted":self._a,"dropped":self._d,"processed":self._p,"depth":self._q.qsize()}

# Drop test
bq = BackpressureQueue2(maxsize=3, strategy="drop")
for i in range(3): assert bq.put(i)
for i in range(5): assert not bq.put(i)
m = bq.metrics(); assert m["accepted"]==3 and m["dropped"]==5, m
for _ in range(3): bq.get(timeout=0.1)
print(f"Drop metrics: {bq.metrics()}")

# Block + producer/consumer test
bq2 = BackpressureQueue2(maxsize=5, strategy="block"); results2=[]
def prod(): [bq2.put(i) for i in range(10)]
def cons(): [results2.append(bq2.get(timeout=2)) for _ in range(10)]
tp,tc = threading.Thread(target=prod), threading.Thread(target=cons)
tp.start(); tc.start(); tp.join(); tc.join()
assert sorted(results2)==list(range(10)); m2=bq2.metrics()
assert m2["accepted"]==10 and m2["dropped"]==0 and m2["processed"]==10
print(f"Block metrics: {m2}")
print("Backpressure ✓")


### Scenario 8 — Idempotent Event Consumer (Exactly-Once Effect)

**Context:** ShopFlow's order service consumes `ORDER_PLACED` events. The network re-delivers the same event after a consumer restart. Implement an `IdempotentConsumer` using a bounded LRU seen-set (simulate Redis `SET NX`). Failed handlers must NOT be cached. Thread-safe; concurrent retries of the same event must not double-execute.


In [ ]:
# -- SOLUTION --
import threading, time
from collections import OrderedDict
from typing import Any, Callable

_SENTINEL2 = object()

class IdempotentConsumer2:
    def __init__(self, max_size=10_000):
        self._max = max_size; self._seen: OrderedDict = OrderedDict(); self._lock = threading.Lock()

    def _set_nx(self, eid, val):
        with self._lock:
            if eid in self._seen: return False
            if len(self._seen) >= self._max: self._seen.popitem(last=False)
            self._seen[eid] = val; return True

    def _get(self, eid): 
        with self._lock: return self._seen.get(eid, _SENTINEL2)

    def _update(self, eid, val):
        with self._lock: self._seen[eid] = val; self._seen.move_to_end(eid)

    def process(self, eid: str, handler: Callable[[], Any]):
        if not self._set_nx(eid, _SENTINEL2):
            cached = self._get(eid)
            while cached is _SENTINEL2: time.sleep(0.001); cached = self._get(eid)
            return cached, True
        try: result = handler()
        except:
            with self._lock: self._seen.pop(eid, None); raise
        self._update(eid, result); return result, False

# Tests
calls = [0]
def charge(): calls[0] += 1; return {"charged": True}

ic = IdempotentConsumer2(max_size=100)
r, dup = ic.process("ord-001", charge); assert not dup and calls[0]==1
r2, dup2 = ic.process("ord-001", charge); assert dup2 and r2==r and calls[0]==1
print(f"Duplicate cached: {r2}, calls={calls[0]}")

# Failure not cached → retry succeeds
try: ic.process("ord-002", lambda: (_ for _ in ()).throw(ValueError("err")))
except ValueError: pass
calls2 = [0]
r3, d3 = ic.process("ord-002", lambda: (calls2.__setitem__(0, calls2[0]+1) or "ok"))
assert not d3 and r3 == "ok"; print(f"Retry after failure: {r3}")

# LRU eviction
sm = IdempotentConsumer2(max_size=3)
for i in range(4): sm.process(f"e{i}", lambda i=i: i)
assert "e0" not in sm._seen and "e3" in sm._seen
print(f"LRU keys: {list(sm._seen.keys())}")

# Concurrent: only 1 execution
cnt = [0]
def slow(): time.sleep(0.05); cnt[0]+=1; return "done"
ic2 = IdempotentConsumer2(); ts=[threading.Thread(target=ic2.process, args=("shared",slow)) for _ in range(5)]
for t in ts: t.start()
for t in ts: t.join()
assert cnt[0]==1; print(f"Concurrent: handler ran {cnt[0]} time ✓")
print("Idempotent consumer ✓")


---
## Scenario-Based Code Questions -- FastAPI Advanced

### Scenario 1 -- ASGI Request ID Middleware

**Context:** Every BuildFast request needs `X-Request-ID` injected (or reuse the caller's ID) and returned in the response.

**Requirements:** Pure ASGI middleware -- no FastAPI import needed.

In [ ]:
# -- SOLUTION --
import uuid
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient
from starlette.datastructures import MutableHeaders

class RequestIDMiddleware:
    def __init__(self, app): self.app = app

    async def __call__(self, scope, receive, send):
        if scope['type'] not in ('http', 'websocket'):
            await self.app(scope, receive, send); return
        headers = dict(scope.get('headers', []))
        req_id  = (headers.get(b'x-request-id') or str(uuid.uuid4()).encode()).decode()
        scope.setdefault('state', {})['request_id'] = req_id

        async def inject(message):
            if message['type'] == 'http.response.start':
                MutableHeaders(scope=message).append('x-request-id', req_id)
            await send(message)

        await self.app(scope, receive, inject)

app = FastAPI()
app.add_middleware(RequestIDMiddleware)

@app.get('/ping')
def ping(request: Request): return {'request_id': request.state.request_id}

client = TestClient(app)
r = client.get('/ping')
assert 'x-request-id' in r.headers
assert r.json()['request_id'] == r.headers['x-request-id']
print('Auto-generated ID:', r.headers['x-request-id'])

r2 = client.get('/ping', headers={'X-Request-ID': 'my-trace-123'})
assert r2.headers['x-request-id'] == 'my-trace-123'
print('Caller-provided ID preserved:', r2.headers['x-request-id'])

---
## Scenario-Based Code Questions -- Concurrency

### Scenario 1 -- Async Connection Pool

**Context:** BuildFast async workers need to reuse N HTTP connections to GitHub instead of creating a new TLS connection per request (~200ms overhead).

**Key:** Use `asyncio.Queue` (not `threading.Queue`) -- it suspends coroutines, not threads.

In [ ]:
# -- SOLUTION --
import asyncio
from contextlib import asynccontextmanager

class FakeConn:
    _n = 0
    def __init__(self): FakeConn._n += 1; self.id = FakeConn._n
    async def get(self, url): await asyncio.sleep(0.001); return {'url': url, 'conn': self.id}

class AsyncConnectionPool:
    def __init__(self, size: int):
        self._size, self._q = size, None

    async def setup(self):
        self._q = asyncio.Queue(maxsize=self._size)
        for _ in range(self._size): await self._q.put(FakeConn())

    @asynccontextmanager
    async def acquire(self):
        conn = await self._q.get()
        try:   yield conn
        finally: await self._q.put(conn)

async def demo():
    pool = AsyncConnectionPool(3)
    await pool.setup()
    async def fetch(i):
        async with pool.acquire() as conn: return await conn.get(f'https://api/{i}')
    results = await asyncio.gather(*[fetch(i) for i in range(10)])
    ids = {r['conn'] for r in results}
    print(f'10 requests via {len(ids)} pooled connections: {sorted(ids)}')
    assert len(ids) <= 3

asyncio.run(demo())

### Scenario 2 -- Thread-Safe Bounded Worker Pool

**Context:** BuildFast limits concurrent GitHub API calls to 3 at a time using a semaphore on top of ThreadPoolExecutor.

In [ ]:
# -- SOLUTION --
import threading, time
from concurrent.futures import ThreadPoolExecutor

class BoundedPool:
    def __init__(self, max_workers: int):
        self._sem  = threading.Semaphore(max_workers)
        self._pool = ThreadPoolExecutor(max_workers=max_workers)

    def submit(self, fn, *args, **kwargs):
        self._sem.acquire()
        def run():
            try:   return fn(*args, **kwargs)
            finally: self._sem.release()  # ALWAYS release
        return self._pool.submit(run)

    def shutdown(self, wait=True): self._pool.shutdown(wait=wait)

peak = {'n': 0, 'max': 0}; lock = threading.Lock()

def task(i):
    with lock: peak['n'] += 1; peak['max'] = max(peak['max'], peak['n'])
    time.sleep(0.05)
    with lock: peak['n'] -= 1
    return f'done_{i}'

pool = BoundedPool(3)
results = [f.result() for f in [pool.submit(task, i) for i in range(12)]]
pool.shutdown()
print(f'All {len(results)} done. Peak concurrency: {peak["max"]} (limit: 3)')
assert peak['max'] <= 3

---
## Scenario-Based Code Questions -- Event-Driven Systems

### Scenario 1 -- Idempotent Event Consumer

**Context:** Kafka delivers `order.placed` at-least-once. Processing an order twice = billing twice. Implement idempotent handling.

**Key insight:** Register the event_id BEFORE processing. If we crash after registration but before processing, we skip on retry -- safer than double-processing.

In [ ]:
# -- SOLUTION --
import threading
from collections import OrderedDict

class IdempotentConsumer:
    def __init__(self, process_fn, max_seen=100_000):
        self._fn, self._max = process_fn, max_seen
        self._seen = OrderedDict()  # LRU-bounded set
        self._lock = threading.Lock()
        self.processed = self.duplicates = 0

    def handle(self, event_id: str, payload: dict) -> bool:
        with self._lock:
            if event_id in self._seen:
                self.duplicates += 1; return False
            self._seen[event_id] = True  # register BEFORE processing
            if len(self._seen) > self._max: self._seen.popitem(last=False)
        self._fn(payload)  # outside lock -- parallel processing
        self.processed += 1
        return True

orders = []
consumer = IdempotentConsumer(lambda p: orders.append(p['order_id']))
events = [('e1',{'order_id':'O1'}),('e2',{'order_id':'O2'}),
          ('e1',{'order_id':'O1'}),('e3',{'order_id':'O3'}),  # e1 duplicate
          ('e2',{'order_id':'O2'})]                            # e2 duplicate
for eid, p in events:
    r = consumer.handle(eid, p)
    print(f'  {eid}: {"processed" if r else "SKIP"}')
assert orders.count('O1') == 1
print(f'Processed: {consumer.processed}, Dupes skipped: {consumer.duplicates}')

---
## Scenario-Based Code Questions -- Networking, Security & Testing

### Scenario 1 -- JWT Sign & Verify from Scratch

**Context:** BuildFast issues JWT tokens. Implement sign/verify using only stdlib to understand the security properties.

**JWT structure:** `base64url(header).base64url(payload).HMAC-SHA256(h.p, secret)`

**Critical:** Use `hmac.compare_digest()` not `==` -- prevents timing attacks.

In [ ]:
# -- SOLUTION --
import base64, hashlib, hmac, json, time

def b64e(data): return base64.urlsafe_b64encode(data).rstrip(b'=').decode()
def b64d(s):
    s += '=' * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s)

def jwt_sign(payload: dict, secret: str) -> str:
    header = b64e(json.dumps({'alg':'HS256','typ':'JWT'}).encode())
    body   = b64e(json.dumps(payload).encode())
    msg    = f'{header}.{body}'.encode()
    sig    = b64e(hmac.new(secret.encode(), msg, hashlib.sha256).digest())
    return f'{header}.{body}.{sig}'

def jwt_verify(token: str, secret: str) -> dict:
    h, b, sig = token.split('.')
    msg = f'{h}.{b}'.encode()
    expected = b64e(hmac.new(secret.encode(), msg, hashlib.sha256).digest())
    if not hmac.compare_digest(expected, sig):  # constant-time!
        raise ValueError('invalid signature')
    payload = json.loads(b64d(b))
    if 'exp' in payload and payload['exp'] < time.time():
        raise ValueError('token expired')
    return payload

SECRET = 'dev-secret'
tok = jwt_sign({'user_id':'u1','role':'admin','exp': time.time()+3600}, SECRET)
payload = jwt_verify(tok, SECRET)
print('Verified:', payload['user_id'], payload['role'])
h, b, s = tok.split('.')
try: jwt_verify(f'{h}.{b}.BADSIG', SECRET)
except ValueError as e: print('Tampered:', e)
old = jwt_sign({'user_id':'u2','exp': time.time()-1}, SECRET)
try: jwt_verify(old, SECRET)
except ValueError as e: print('Expired:', e)

### Scenario 2 -- SQL Injection: Vulnerable vs Safe

**Context:** BuildFast lets users search build logs by commit message. Show the VULNERABLE pattern and the SAFE parameterized fix.

In [ ]:
# -- SOLUTION --
import sqlite3

db = sqlite3.connect(':memory:')
db.execute('CREATE TABLE builds (id INT, commit_msg TEXT, status TEXT)')
db.executemany('INSERT INTO builds VALUES (?,?,?)',
               [(1,'fix: auth bug','success'),(2,'feat: new UI','failed')])
db.commit()

# VULNERABLE -- string interpolation
def search_VULN(commit_msg: str):
    sql = f"SELECT * FROM builds WHERE commit_msg LIKE '%{commit_msg}%'"
    return db.execute(sql).fetchall()

injection = "' OR '1'='1"
rows = search_VULN(injection)
print(f'[VULN] Injection returned {len(rows)} rows (ALL rows exposed!)')

# SAFE -- parameterized query
def search_SAFE(commit_msg: str):
    return db.execute('SELECT * FROM builds WHERE commit_msg LIKE ?',
                      (f'%{commit_msg}%',)).fetchall()

safe_rows = search_SAFE(injection)
print(f'[SAFE] Injection returned {len(safe_rows)} rows (0 = blocked |/)')
real_rows = search_SAFE('fix:')
print(f'[SAFE] Legitimate search: {real_rows}')

---
## Scenario-Based Code Questions -- ELK & Observability

### Scenario 1 -- Structured JSON Logger with Correlation ID

**Context:** BuildFast needs every log line to be valid JSON parseable by Elasticsearch. Include `timestamp`, `level`, `service`, `request_id`, `message`.

In [ ]:
# -- SOLUTION --
import json, time, uuid

class StructuredLogger:
    def __init__(self, service: str):
        self.service, self._request_id = service, None

    def bind(self, request_id: str):
        child = StructuredLogger(self.service)
        child._request_id = request_id
        return child

    def _log(self, level, msg, **extra):
        print(json.dumps({
            'timestamp':  time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
            'level':      level, 'service': self.service,
            'request_id': self._request_id or 'no-request',
            'message':    msg, **extra
        }))

    def info(self, m, **k):  self._log('INFO', m, **k)
    def warn(self, m, **k):  self._log('WARN', m, **k)
    def error(self, m, **k): self._log('ERROR', m, **k)

log = StructuredLogger('build-service')
req = log.bind(request_id=str(uuid.uuid4())[:8])
req.info('Pipeline started', pipeline='frontend-ci', user='alice@co.com')
req.warn('Slow step', step='lint', duration_ms=8900)
req.error('Step failed', step='test', exit_code=1)

---
## Scenario-Based Code Questions -- Database Scaling

### Scenario 1 -- Read Replica Query Router

**Context:** ShopFlow's product catalogue: 95% reads, 5% writes. Route SELECT queries to the replica, writes to primary. Detect operation type from SQL prefix.

In [ ]:
# -- SOLUTION --
import re

class QueryRouter:
    WRITE_RE = re.compile(
        r'^\s*(INSERT|UPDATE|DELETE|CREATE|DROP|ALTER|TRUNCATE)',
        re.IGNORECASE
    )

    def __init__(self, primary: str, replica: str):
        self.primary, self.replica = primary, replica

    def route(self, sql: str) -> str:
        return self.primary if self.WRITE_RE.match(sql) else self.replica

router = QueryRouter('postgres://primary:5432/db', 'postgres://replica:5432/db')
cases = [
    ('SELECT * FROM products WHERE id = $1', 'replica'),
    ('INSERT INTO orders VALUES ($1, $2)',    'primary'),
    ('UPDATE inventory SET qty = $1',         'primary'),
    ('SELECT COUNT(*) FROM builds',           'replica'),
]
for sql, expected in cases:
    t = router.route(sql)
    assert expected in t
    print(f'  {"WRITE" if expected=="primary" else "READ ":5s}: {sql[:50]}')
print('All routing correct |/')